# 📋 Codeframe — AI-Assisted Qualitative Coding Toolkit

A complete end-to-end notebook for coding open-ended survey responses.  
All sections share a **single engine**: one setup, one API key, one output format.

---

## How to use this notebook

```
SECTION 0  ·  Setup           ← Run once at the start of every session
SECTION 1  ·  Preprocessing   ← Run once per dataset

Then choose your coding path:

  SECTION 2  ·  AI Codebook        No codebook? Let AI discover themes (UMAP + HDBSCAN + configured LLM)
  SECTION 3  ·  User Codebook      Have codes already? Import and apply them

Then refine as needed (both are optional and chainable):

  SECTION 4  ·  Split a Code       Code too broad? Break it into finer sub-codes
  SECTION 5  ·  Merge Codes        Codes too similar? Collapse them into one
```

### Data contract
Every section that produces output returns a **(df, codebook)** pair.  
Pass the output of any section directly into Section 4 or 5 to keep refining.  
Sections 4 and 5 update `df_current` and `codebook_current` in place so you  
can chain them without extra bookkeeping.

### Output files (every section)
- **Excel workbook** — Résumé / Fréquences / Co-occurrence / Codebook / Données codées  
- **Frequency bar chart** (.png)  
- **Co-occurrence heatmap** (.png)




---
## SECTION 0 · Setup
*Run every cell in this section once at the start of each session.*  
*All other sections depend on these definitions.*

| Cell | Purpose |
|------|---------|
| **0-A** | Install dependencies |
| **0-B** | Project configuration — **edit this cell for each new project** |
| **0-C** | Imports and logging |
| **0-D** | Constants (stop-words, regex, model names) |
| **0-E** | Shared utility functions |
| **0-F** | Reporting engine (charts + Excel export) |


### 0-A · Dependencies
*Comment out after the first install to speed up session startup.*

In [100]:
# pip install pandas numpy scikit-learn
# pip install openai matplotlib seaborn openpyxl
# pip install umap-learn hdbscan ftfy lingua-language-detector tenacity pydantic tqdm


### 0-B · Project configuration
**This is the only cell you need to edit when starting a new project.**  
All downstream cells read from these variables — never hard-code paths elsewhere.


Set `CODEBOOK_LANGUAGE_CODE = "fr"` for a French codebook or `CODEBOOK_LANGUAGE_CODE = "en"` for an English codebook. This controls generated code names, definitions, enrichment, split sub-codes, merged-code names, and the catch-all label.


This version also retries codebook generation automatically if the first LLM response is cut off before producing complete JSON.


In [101]:
import os
from pathlib import Path

# ── API ────────────────────────────────────────────────────────────────────────────────
# SECURITY + REPRODUCIBILITY

from dotenv import load_dotenv
load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

# ── Survey data ──────────────────────────────────────────────────────────────
SURVEY_FILE   = "PREF1_DEGUSTATION_AM_FESTI237.xlsx"
SURVEY_SHEET  = 0                            # sheet name or 0-based index
SURVEY_COL    = "feedback"                   # column with open-ended responses
FILE_STEM     = Path(SURVEY_FILE).stem        # used to name output files

# ── Survey context ──────────────────────────────────────────────────────────
SURVEY_QUESTION = "Pourquoi préférez-vous cette boisson ?"
SURVEY_CONTEXT = f"""
Les répondants sont des consommateurs de boissons au Cameroun.
Ils répondent à la question ouverte : « {SURVEY_QUESTION} »
L'objectif est d'identifier les thèmes liés à ce que les consommateurs
apprécient (goût, fraîchissement, prix, marque, bénéfices, habitudes).
"""

# ── Language detection + translation settings ────────────────────────────────
# Default behavior:
#   1) Detect response language with Lingua
#   2) Translate detected source-language responses into the selected target language
#
# To translate a different detected source language, change SOURCE_LANGUAGE_CODE.
# To translate into a different target language, change TARGET_LANGUAGE_CODE.
SOURCE_LANGUAGE_CODE = "en"     # default source detected by Lingua
TARGET_LANGUAGE_CODE = "fr"     # default target translation language

LANGUAGE_NAME_BY_CODE = {
    "en": "English",
    "fr": "French",
    "es": "Spanish",
    "de": "German",
    "it": "Italian",
    "pt": "Portuguese",
    "nl": "Dutch",
    "ar": "Arabic",
}

SOURCE_LANGUAGE_NAME = LANGUAGE_NAME_BY_CODE.get(SOURCE_LANGUAGE_CODE, SOURCE_LANGUAGE_CODE)
TARGET_LANGUAGE_NAME = LANGUAGE_NAME_BY_CODE.get(TARGET_LANGUAGE_CODE, TARGET_LANGUAGE_CODE)

# ── Codebook output language ────────────────────────────────────────────────
# Controls the language used for AI-generated:
#   - Section 2 code names + definitions
#   - Section 3 AI-enriched definitions
#   - Section 4 AI-proposed sub-code names + definitions
#   - Section 5 AI-proposed merged-code names + definitions
# Use "fr" for French or "en" for English.
#
# This is intentionally separate from TARGET_LANGUAGE_CODE:
#   TARGET_LANGUAGE_CODE controls the language of the text used for coding.
#   CODEBOOK_LANGUAGE_CODE controls the language of the codebook/report labels.
CODEBOOK_LANGUAGE_CODE = "fr"   # change to "en" for an English codebook

SUPPORTED_CODEBOOK_LANGUAGE_CODES = {"fr", "en"}
CODEBOOK_LANGUAGE_CODE = str(CODEBOOK_LANGUAGE_CODE).lower().strip()
if CODEBOOK_LANGUAGE_CODE not in SUPPORTED_CODEBOOK_LANGUAGE_CODES:
    raise ValueError(
        f'Unsupported CODEBOOK_LANGUAGE_CODE="{CODEBOOK_LANGUAGE_CODE}". '
        'Use "fr" or "en".'
    )

CODEBOOK_LANGUAGE_NAME = LANGUAGE_NAME_BY_CODE.get(CODEBOOK_LANGUAGE_CODE, CODEBOOK_LANGUAGE_CODE)

# Text column name created by Section 1 and used by downstream coding sections.
# This automatically follows the selected target language.
TEXT_COL = f"final_text_{TARGET_LANGUAGE_CODE}"

# ── Models ───────────────────────────────────────────────────────────────────────────────
# Two-model strategy: match model capability to task complexity.
#
# REASONING_MODEL — low-frequency calls that shape analytical structure
#   codebook generation, sub-code proposal, merge naming
#   Keep this stronger than CODING_MODEL when quality matters most.
REASONING_MODEL = "gpt-5.4-mini"

# CODING_MODEL — high-volume repetitive calls (most of your token spend)
#   translation, response coding, code enrichment
#   Use your cheapest reliable model for classification-style tasks.
#   If this model is unavailable in your account, replace it with your approved model.
CODING_MODEL    = "gpt-4.1-mini"

# Embedding model (OpenAI API)
# text-embedding-3-small returns 1536-dimensional embeddings by default.
EMBEDDING_MODEL = "text-embedding-3-small"
EMBEDDING_BATCH_SIZE = 100

# ── Quick override ───────────────────────────────────────────────────────────────
# Run everything on one model (uncomment one line):
# REASONING_MODEL = CODING_MODEL = 'gpt-4o-mini'   # cheaper, for dev/testing
# REASONING_MODEL = CODING_MODEL = 'gpt-5.4-mini'  # uniform high quality if available


# ── API reliability settings ─────────────────────────────────────────────────────────
# Increase timeout because coding batches can be slow when the codebook is large.
OPENAI_TIMEOUT_SECONDS = 180.0
OPENAI_MAX_RETRIES_IN_WRAPPER = 10

# ── Cost-control settings ───────────────────────────────────────────────────────────────
# Response coding cost is driven mostly by repeated codebook prompts.
# Larger batches repeat the codebook fewer times. If JSON parsing becomes unstable,
# reduce CODING_BATCH_SIZE to 15 or 10.
CODING_BATCH_SIZE      = 10   # responses per coding call  (Sections 2, 3, 4). Use 10 to reduce API timeouts.
TRANSLATION_BATCH_SIZE = 10   # EN responses per translation call (Section 1). Use 10 to reduce API timeouts.

# Use a compact codebook in high-volume coding calls. The full codebook renderer is
# preserved for documentation and enrichment, but coding usually only needs names + definitions.
USE_COMPACT_CODEBOOK_FOR_CODING = True

# Keep response-coding output small. Set True only for debugging; it increases output tokens.
SAVE_CODING_REASONING = False

# Section 2 codebook generation guardrails.
# CODEBOOK_GENERATION_MAX_TOKENS limits the first-attempt answer length for the codebook call.
# CODEBOOK_GENERATION_RETRY_MAX_TOKENS is used automatically when the first answer is cut off.
#
# CODEBOOK_TARGET_CODES_CAP is optional. By default it is None so the number of
# generated codes follows the clustering result: target_codes = len(representatives).
# Set it to an integer such as 12 only when you intentionally want to compress
# many clusters into fewer higher-level themes for cost-control or executive summaries.
# Keeping it as None preserves the clustering-derived granularity.
CODEBOOK_GENERATION_MAX_TOKENS       = 8000   # first attempt; 3000 was too small for larger codebooks
CODEBOOK_GENERATION_RETRY_MAX_TOKENS = 12000  # automatic retry budget if JSON is truncated
CODEBOOK_TARGET_CODES_CAP            = None   # None = cluster-derived code count; integer = optional cap


# ── Reset / checkpoint settings ───────────────────────────────────────────
# Sections 2 and 3 automatically create baseline checkpoints after coding.
# Sections 4 and 5 automatically create pre/post checkpoints around split/merge.
# These checkpoints define what "reset" means and allow recovery after a kernel restart.
CHECKPOINT_DIR = Path('checkpoints') / FILE_STEM
SAVE_RESET_CHECKPOINTS_TO_DISK = True

# ── Clustering parameters (Section 2 only) ──────────────────────────────────────
CLUSTER_UMAP_COMPONENTS = 10
CLUSTER_UMAP_NEIGHBORS  = 15
CLUSTER_MIN_SAMPLES     = 1
CLUSTER_EPSILON         = 0.4
CLUSTER_REPRESENTATIVES = 3              # fewer examples per cluster = smaller prompts


### 0-C · Imports and logging

In [102]:
import os, re, time, json, logging, warnings, html, unicodedata, random, copy
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import ftfy
from lingua import Language, LanguageDetectorBuilder
from pathlib import Path
from datetime import datetime, timezone
from openai import OpenAI
from pydantic import BaseModel
from tenacity import retry, stop_after_attempt, wait_exponential
from tqdm import tqdm

logging.basicConfig(level=logging.INFO,
                    format="%(asctime)s - %(levelname)s - %(message)s",
                    datefmt="%H:%M:%S")
logger = logging.getLogger(__name__)
warnings.filterwarnings('ignore')

class TranslationResult(BaseModel):
    translated_text: str


### 0-D · Constants

In [103]:
# Phrases treated as empty / non-codeable responses (FR + EN)
EMPTY_RESPONSES = {
    'n/a','na','none','no','yes','idk','dk','dont know',"don't know",
    'nothing','no comment','no comments','-','.','..','...','nil',
    'not applicable','no answer','no response','blank','skip','skipped',
    '','aucun','aucune','rien','néant','sans','—',
    "d'accord",'ras','r.a.s','pas de commentaire','aucun commentaire',
    'je ne sais pas','je sais pas','rien à signaler','rien a signaler'
}
DEFAULT_MIN_WORDS = 2  # responses shorter than this are discarded

# Compiled once; reused in every clean_text() call
URL_RE      = re.compile(r'\bhttps?://[^\s<>]+|\bwww\.[^\s<>]+', re.IGNORECASE)
EMAIL_RE    = re.compile(r'\b[\w.\-+]+@[\w.\-]+\.\w+\b',       re.IGNORECASE)
# HTML_TAG_RE = re.compile(r'<[^>]+')
HTML_TAG_RE = re.compile(r'<[^>]+>')
EMOJI_RE    = re.compile(
    r'[\U0001F600-\U0001F64F\U0001F300-\U0001F5FF\U0001F680-\U0001F6FF'
    r'\U0001F700-\U0001F77F\U0001F780-\U0001F7FF\U0001F800-\U0001F8FF'
    r'\U0001F900-\U0001F9FF\U0001FA00-\U0001FA6F\U0001FA70-\U0001FAFF'
    r'\U00002702-\U000027B0\U000024C2-\U0001F251\U0001F1E6-\U0001F1FF'
    r'\U00002600-\U000026FF]|\uFE0F|\U0001F3FB|\U0001F3FC|\U0001F3FD'
    r'|\U0001F3FE|\U0001F3FF', re.UNICODE)


### 0-E · Shared utility functions
*Defined once here. Never copy-pasted into individual sections.*

| Function | Model used | Purpose |
|----------|------------|---------|
| `clean_text` | — | Sanitise a response string |
| `is_valid_response` | — | Filter junk / empty responses |
| `_extract_json` | — | Parse JSON from LLM output |
| `_validate_codebook` | — | Assert required fields exist |
| `chat_with_retry` | either | Rate-limit-safe wrapper; logs token usage; fails fast on quota errors |
| `get_code_columns` | — | List `(name, code_col, conf_col)` tuples |
| `format_codebook_for_prompt` | — | Render full codebook as LLM text |
| `format_codebook_for_prompt_compact` | — | Render compact codebook for high-volume coding |
| `add_codes_column` | — | Rebuild `codes` / `num_codes` / `has_multi_label` |
| `display_codebook` | — | Pretty-print a codebook |
| `save_codebook` / `load_codebook` | — | Persist a codebook dict as JSON |
| `enrich_codebook_with_ai` | **CODING_MODEL** | Enrich all undefined codes in 1 call |
| `_deduplicate_texts` | — | Skip re-coding identical responses |
| `_translate_batch` | **CODING_MODEL** | Translate N responses per API call |

> **Model routing — where each model is used**
>
> | Task | Model | Calls per run | Why |
> |------|-------|---------------|-----|
> | Codebook generation | **REASONING_MODEL** | 1 | Shapes entire analytical structure |
> | Sub-code proposal | **REASONING_MODEL** | 1 per split | Conceptual judgement |
> | Merge name proposal | **REASONING_MODEL** | 1 per merge | Semantic synthesis |
> | Translation | **CODING_MODEL** | ceil(n\_EN / 20) | Simple EN→FR |
> | Response coding | **CODING_MODEL** | ceil(n / `CODING_BATCH_SIZE`) | Structured classification using compact prompts |
> | Code enrichment | **CODING_MODEL** | 1 per section | Definition writing |


> **Cost note**  
> The biggest token driver is repeated response coding. Each coding batch includes the codebook, so increasing `CODING_BATCH_SIZE` and using `USE_COMPACT_CODEBOOK_FOR_CODING=True` usually has the largest cost impact.


In [104]:
# ── LLM infrastructure ───────────────────────────────────────────────────────

# Every successful OpenAI call appends one row here. This makes token/cost hotspots visible.
LLM_USAGE_LOG = []

def _append_llm_usage(resp, *, model, messages=None):
    """
    Record token usage from an OpenAI response if the API returned usage metadata.

    Usage fields vary slightly by SDK/model, so this helper is defensive.
    The log is intentionally lightweight and does not store prompt text.
    """
    usage = getattr(resp, 'usage', None)
    if usage is None:
        logger.info(f'LLM usage unavailable | model={model}')
        return

    prompt_tokens     = getattr(usage, 'prompt_tokens', None)
    completion_tokens = getattr(usage, 'completion_tokens', None)
    total_tokens      = getattr(usage, 'total_tokens', None)

    # Some OpenAI responses include cached token details under prompt_tokens_details.
    details = getattr(usage, 'prompt_tokens_details', None)
    cached_tokens = getattr(details, 'cached_tokens', None) if details is not None else None

    prompt_chars = None
    if messages:
        prompt_chars = sum(len(str(m.get('content', ''))) for m in messages)

    row = {
        'timestamp': datetime.now(timezone.utc).isoformat(),
        'model': model,
        'prompt_tokens': prompt_tokens,
        'completion_tokens': completion_tokens,
        'total_tokens': total_tokens,
        'cached_prompt_tokens': cached_tokens,
        'prompt_chars_approx': prompt_chars,
    }
    LLM_USAGE_LOG.append(row)

    logger.info(
        'LLM usage | model=%s | prompt=%s | completion=%s | total=%s | cached=%s',
        model, prompt_tokens, completion_tokens, total_tokens, cached_tokens
    )

def get_llm_usage_log():
    """Return token usage collected during the current notebook session."""
    return pd.DataFrame(LLM_USAGE_LOG)

_REASONING_CHAT_MODEL_PREFIXES = ("o1", "o3", "o4", "gpt-5")


def _uses_max_completion_tokens(model):
    """
    Return True for Chat Completions models that require max_completion_tokens.

    The notebook keeps max_tokens as its internal argument name so existing calls do
    not need to change. This helper only controls the outbound OpenAI API payload.
    """
    model_name = str(model or "").lower().strip()
    return model_name.startswith(_REASONING_CHAT_MODEL_PREFIXES)


def _build_chat_completion_kwargs(*, model, messages, temperature, max_tokens):
    """
    Build a Chat Completions request with model-compatible token-limit naming.

    Newer/reasoning Chat Completions models expect max_completion_tokens.
    Earlier/non-reasoning Chat Completions models may still expect max_tokens.
    """
    kwargs = {
        'model': model,
        'messages': messages,
    }

    if temperature is not None:
        kwargs['temperature'] = temperature

    if max_tokens is not None:
        token_param = ('max_completion_tokens'
                       if _uses_max_completion_tokens(model)
                       else 'max_tokens')
        kwargs[token_param] = max_tokens

    return kwargs


def _retry_with_alternate_chat_params(client, request_kwargs, error):
    """
    Retry one non-transient BadRequest caused by OpenAI parameter compatibility.

    This protects the notebook when switching between model families without
    changing every downstream function call.
    """
    msg = str(error).lower()
    retry_kwargs = dict(request_kwargs)
    changed = False

    if 'max_tokens' in msg and 'max_completion_tokens' in msg:
        if 'max_tokens' in retry_kwargs:
            retry_kwargs['max_completion_tokens'] = retry_kwargs.pop('max_tokens')
            changed = True
        elif 'max_completion_tokens' in retry_kwargs:
            retry_kwargs['max_tokens'] = retry_kwargs.pop('max_completion_tokens')
            changed = True

    if 'temperature' in msg and ('unsupported' in msg or 'does not support' in msg):
        if 'temperature' in retry_kwargs:
            retry_kwargs.pop('temperature', None)
            changed = True

    if not changed:
        raise error

    logger.warning('Retrying OpenAI call with compatibility-adjusted parameters.')
    return client.chat.completions.create(**retry_kwargs)


def chat_with_retry(client, *, model, messages, temperature, max_tokens,
                    max_retries=None, base_delay=2.0, max_delay=90.0,
                    log_usage=True):
    """
    Rate-limit-safe OpenAI wrapper with quota-aware fail-fast logic and token logging.

    Retry behaviour
    ───────────────
    Retried  (transient) : HTTP 429 rate-limit, 5xx, timeout
    NOT retried (fatal)  : insufficient_quota (billing), invalid_api_key

    Parameter compatibility
    ───────────────────────
    The notebook uses max_tokens internally, then maps it to the correct OpenAI
    Chat Completions parameter for the selected model:
      - reasoning/newer chat models: max_completion_tokens
      - earlier/non-reasoning chat models: max_tokens

    A one-time compatibility fallback is also applied for unsupported token-limit
    parameter names or unsupported temperature settings.

    Cost visibility
    ───────────────
    On success, usage metadata is appended to LLM_USAGE_LOG.
    Run get_llm_usage_log() after a section to see prompt/completion/total tokens by call.
    """
    if max_retries is None:
        max_retries = globals().get('OPENAI_MAX_RETRIES_IN_WRAPPER', 10)

    QUOTA_CODES = {'insufficient_quota', 'quota_exceeded',
                   'billing_not_active', 'account_deactivated'}
    last_err = None

    for attempt in range(max_retries):
        request_kwargs = _build_chat_completion_kwargs(
            model=model,
            messages=messages,
            temperature=temperature,
            max_tokens=max_tokens,
        )
        request_kwargs['timeout'] = globals().get('OPENAI_TIMEOUT_SECONDS', 180.0)

        try:
            resp = client.chat.completions.create(**request_kwargs)
            if log_usage:
                _append_llm_usage(resp, model=model, messages=messages)
            return resp

        except Exception as e:
            last_err = e
            status = (getattr(e, 'status_code', None) or
                      getattr(getattr(e, 'response', None), 'status_code', None))
            body   = getattr(e, 'body', None) or {}
            code   = (body.get('error', {}).get('code') or
                      getattr(e, 'code', None) or '')
            msg    = str(e).lower()

            if code in QUOTA_CODES or 'insufficient_quota' in msg:
                raise RuntimeError(
                    '\n\n'
                    '╭──────────────────────────────────────────────────────╮\n'
                    '│  OpenAI quota exhausted — account has no credits     │\n'
                    '├──────────────────────────────────────────────────────┤\n'
                    '│  1. https://platform.openai.com/account/billing       │\n'
                    '│  2. Add credits, then re-run this cell                │\n'
                    '╰──────────────────────────────────────────────────────╯'
                ) from e

            if status == 401 or 'invalid_api_key' in msg:
                raise RuntimeError(
                    'Invalid API key. Set OPENAI_API_KEY as an environment variable.') from e

            # BadRequest compatibility fallback: retry immediately once with the
            # alternate parameter style instead of treating it as a transient error.
            if status == 400:
                try:
                    resp = _retry_with_alternate_chat_params(client, request_kwargs, e)
                    if log_usage:
                        _append_llm_usage(resp, model=model, messages=messages)
                    return resp
                except Exception as fallback_error:
                    last_err = fallback_error
                    raise

            retryable = status in (429, 500, 502, 503, 504) or any(
                kw in msg for kw in ('rate limit', 'too many requests',
                                     'timeout', 'temporarily'))
            if not retryable or attempt == max_retries - 1:
                raise

            delay = min(max_delay, base_delay * (2 ** attempt))
            delay += random.uniform(0, 0.5 * delay)
            logger.warning(f'Retry {attempt+1}/{max_retries} after '
                           f'{delay:.1f}s (status={status})')
            time.sleep(delay)

    raise last_err


# ── Codebook helpers ─────────────────────────────────────────────────────────

CODE_ID_FIELD = 'code_id'


def _clean_code_id(value):
    """Return a stable string code ID, preserving user-supplied IDs when present."""
    if value is None:
        return ''
    try:
        if pd.isna(value):
            return ''
    except Exception:
        pass
    if isinstance(value, float) and value.is_integer():
        return str(int(value))
    return str(value).strip()


def _clean_code_name(value):
    """Return a stripped code name string."""
    if value is None:
        return ''
    try:
        if pd.isna(value):
            return ''
    except Exception:
        pass
    return str(value).strip()


def _clean_optional_text(value):
    """Return a stripped optional text value, converting pandas/Excel blanks to ''."""
    if value is None:
        return ''
    try:
        if pd.isna(value):
            return ''
    except Exception:
        pass
    return str(value).strip()




def _safe_confidence(value, default=0.0):
    """
    Return a numeric confidence in [0.0, 1.0].

    LLMs sometimes return confidence as a string (for example "0.8") or an
    invalid value. Keeping confidence columns strictly numeric prevents pandas
    from silently upcasting them to object dtype and protects downstream sums,
    max operations, and Excel reporting.
    """
    try:
        if value is None or value == '':
            value = default
        value = float(value)
    except (TypeError, ValueError):
        value = float(default)
    if not np.isfinite(value):
        value = float(default)
    return max(0.0, min(1.0, value))


def _normalized_code_name_set(values):
    """Return an order-insensitive set of cleaned code names."""
    if values is None:
        return set()
    if isinstance(values, str):
        values = [values]
    return {_clean_code_name(v) for v in values if _clean_code_name(v)}


def _same_code_name_set(left, right):
    """Compare two code-name lists after stripping blanks and ignoring order."""
    return _normalized_code_name_set(left) == _normalized_code_name_set(right)


def _canonical_code_key(value):
    """
    Canonical key used to match AI-returned code names back to codebook entries.

    The LLM may return either "Name" or "[ID] Name". This helper strips an
    optional leading [ID] prefix and normalizes whitespace/case so enrichment
    does not silently fail because of formatting differences.
    """
    text = _clean_code_name(value)
    text = re.sub(r'^\[[^\]]+\]\s*', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text.casefold()


def _codebook_language_code(language_code=None):
    """Return the configured codebook language code. Currently supports French and English."""
    if language_code is None:
        language_code = globals().get('CODEBOOK_LANGUAGE_CODE', 'fr')
    code = str(language_code or 'fr').lower().strip()
    return code if code in {'fr', 'en'} else 'fr'


def _codebook_language_name(language_code=None):
    """Human-readable name for the configured codebook language."""
    code = _codebook_language_code(language_code)
    return globals().get('LANGUAGE_NAME_BY_CODE', {}).get(code, code)


def _codebook_is_english(language_code=None):
    return _codebook_language_code(language_code) == 'en'


def _codebook_text(fr_text, en_text, language_code=None):
    """Return French or English text according to CODEBOOK_LANGUAGE_CODE."""
    return en_text if _codebook_is_english(language_code) else fr_text


def get_default_other_code_name(language_code=None):
    """Default catch-all code label in the selected codebook language."""
    return _codebook_text('Autre / Non classifiable',
                          'Other / Not classifiable',
                          language_code)


def _default_other_code_metadata(other_name=None, language_code=None):
    """Default catch-all definition and criteria in the selected codebook language."""
    if other_name is None:
        other_name = get_default_other_code_name(language_code)

    if _codebook_is_english(language_code):
        return {
            'definition': 'Valid response that does not clearly match any other code in the codebook.',
            'inclusion_criteria': [
                'Use only when no specific code applies.',
                'Use for ambiguous, off-topic, or non-classifiable responses.'
            ],
            'exclusion_criteria': [
                'Do not use if a more specific code applies.',
                'Do not use for blank or invalid responses already removed during preprocessing.'
            ],
        }

    return {
        'definition': (
            'Réponse valide qui ne correspond clairement à aucun autre code '
            'du livre de codes.'
        ),
        'inclusion_criteria': [
            "Utiliser seulement lorsqu'aucun code spécifique ne s'applique.",
            'Utiliser pour les réponses ambiguës, hors sujet ou non classifiables.'
        ],
        'exclusion_criteria': [
            "Ne pas utiliser si un code plus spécifique s'applique.",
            'Ne pas utiliser pour les réponses vides ou invalides déjà retirées au prétraitement.'
        ],
    }


def _fallback_code_definition(code_name, *, survey_context='', parent_code='', merged_from=None, language_code=None):
    """
    Deterministic fallback definition for new split/merge-derived codes.

    This is only used when manual definition is blank and AI enrichment fails to
    return a usable definition. It keeps the governed codebook complete while
    remaining transparent and conservative.
    """
    name = _clean_code_name(code_name)
    ctx = _clean_optional_text(survey_context)
    parent = _clean_code_name(parent_code)

    if _codebook_is_english(language_code):
        if merged_from:
            merged = ', '.join(_clean_code_name(x) for x in merged_from if _clean_code_name(x))
            base = f'Responses grouped under "{name}", consolidating ideas previously coded as: {merged}.'
        elif parent:
            base = f'Responses related to the sub-theme "{name}" within the parent code "{parent}".'
        else:
            base = f'Responses associated with the theme "{name}".'
        if ctx:
            return base + ' Use when the verbatim clearly matches this theme in the study context.'
        return base + ' Use when the verbatim clearly matches this theme.'

    if merged_from:
        merged = ', '.join(_clean_code_name(x) for x in merged_from if _clean_code_name(x))
        base = f'Réponses regroupées sous « {name} », consolidant les idées auparavant codées comme : {merged}.'
    elif parent:
        base = f'Réponses relevant du sous-thème « {name} » dans le code parent « {parent} ».'
    else:
        base = f'Réponses associées au thème « {name} ».'
    if ctx:
        return base + ' À utiliser lorsque le verbatim correspond clairement à ce thème dans le contexte de l’étude.'
    return base + ' À utiliser lorsque le verbatim correspond clairement à ce thème.'


def _apply_definition_fallbacks(codebook, *, survey_context='', parent_code='', merged_from=None, reason='fallback'):
    """
    Fill any remaining blank definitions with deterministic definitions.

    Existing manual or AI-generated definitions are preserved. A metadata flag is
    added so reviewers can identify fallback-generated definitions later.
    """
    filled = []
    for code in codebook.get('codes', []):
        if not _clean_optional_text(code.get('definition')):
            code['definition'] = _fallback_code_definition(
                code.get('name', ''),
                survey_context=survey_context,
                parent_code=parent_code,
                merged_from=merged_from,
            )
            code['definition_source'] = reason
            if not code.get('inclusion_criteria'):
                if _codebook_is_english():
                    code['inclusion_criteria'] = [
                        f'The verbatim clearly matches the theme "{code.get("name", "")}".',
                        'The main meaning of the response is consistent with this category.'
                    ]
                else:
                    code['inclusion_criteria'] = [
                        f'Le verbatim correspond clairement au thème « {code.get("name", "")} ».',
                        'Le sens principal de la réponse est cohérent avec cette catégorie.'
                    ]
            if not code.get('exclusion_criteria'):
                if _codebook_is_english():
                    code['exclusion_criteria'] = [
                        'Do not use if a more specific or more appropriate code applies.',
                        'Do not use if the verbatim is ambiguous or not clearly related to this theme.'
                    ]
                else:
                    code['exclusion_criteria'] = [
                        'Ne pas utiliser si un code plus spécifique ou plus approprié s’applique.',
                        'Ne pas utiliser si le verbatim est ambigu ou sans lien clair avec ce thème.'
                    ]
            if not code.get('example_verbatims'):
                code['example_verbatims'] = []
            filled.append(code.get('name', ''))
    if filled:
        logger.warning('Filled missing definitions with deterministic fallback for: %s', filled)
    return codebook


def codebook_has_code_ids(codebook):
    """True when the codebook is governed by a user-supplied code ID column."""
    return any(_clean_code_id(c.get(CODE_ID_FIELD, '')) for c in codebook.get('codes', []))


def codebook_requires_manual_ids(codebook):
    """
    True when any existing code has an ID.

    In that case, every new code must also receive an ID. The ID can be supplied
    manually or generated automatically by the notebook for split/merge-derived
    codes. Existing user-provided IDs are never changed.
    """
    return codebook_has_code_ids(codebook)


def _existing_code_ids(codebook):
    """Return all non-blank code IDs currently present in a codebook."""
    return {
        _clean_code_id(c.get(CODE_ID_FIELD, ''))
        for c in codebook.get('codes', [])
        if _clean_code_id(c.get(CODE_ID_FIELD, ''))
    }


def _slugify_for_code_id(value, default='CODE', max_len=48):
    """
    Convert a code name or prefix into a stable, readable code-ID token.

    The result is uppercase ASCII with underscores only. This is deterministic so
    rerunning the same split/merge with the same names produces the same base ID;
    a numeric suffix is added only when needed to avoid collisions.
    """
    import unicodedata as _ud
    text = _clean_code_name(value)
    text = _ud.normalize('NFKD', text).encode('ascii', 'ignore').decode('ascii')
    text = re.sub(r'[^A-Za-z0-9]+', '_', text).strip('_').upper()
    text = re.sub(r'_+', '_', text)
    if not text:
        text = default or ''
    text = text[:max_len].strip('_')
    return text or default or ''


def _make_unique_code_id(base, existing_ids, max_len=64):
    """Return a unique code ID, appending _02, _03, ... if needed."""
    base = _slugify_for_code_id(base, default='CODE', max_len=max_len)
    if not base:
        base = 'CODE'
    candidate = base
    i = 2
    while candidate in existing_ids:
        suffix = f'_{i:02d}'
        candidate = f'{base[:max_len-len(suffix)].rstrip("_")}{suffix}'
        i += 1
    existing_ids.add(candidate)
    return candidate


def _auto_code_id_for_new_code(codebook, code_name, *, prefix='', existing_ids=None):
    """
    Generate a stable, collision-checked ID for a new split/merge-derived code.

    Examples:
      parent ID PRIX + sub-code "Prix trop élevé" -> PRIX_PRIX_TROP_ELEVE
      merged code "Bénéfices fonctionnels"       -> MERGE_BENEFICES_FONCTIONNELS
    """
    if existing_ids is None:
        existing_ids = _existing_code_ids(codebook)
    prefix = _slugify_for_code_id(prefix, default='', max_len=24) if prefix else ''
    name_part = _slugify_for_code_id(code_name, default='CODE', max_len=40)
    base = f'{prefix}_{name_part}' if prefix else name_part
    return _make_unique_code_id(base, existing_ids)


def _normalize_manual_id_list(values, expected_len, label):
    """
    Normalize optional manual IDs.

    Empty list/None means "auto-generate where IDs are required". A non-empty
    list must align with the code names, but individual items may be blank so the
    notebook can auto-generate only the missing IDs.
    """
    values = values or []
    ids = [_clean_code_id(x) for x in values]
    if ids and len(ids) != expected_len:
        raise ValueError(f'{label} must be blank or have exactly {expected_len} item(s).')
    if not ids:
        ids = [''] * expected_len
    return ids


def _code_ref(code):
    """Human-readable code reference for prompts/reports."""
    cid = _clean_code_id(code.get(CODE_ID_FIELD, ''))
    name = _clean_code_name(code.get('name', ''))
    return f'[{cid}] {name}' if cid else name


def make_code_entry(name, *, code_id='', definition='', inclusion_criteria=None,
                    exclusion_criteria=None, example_verbatims=None, **extra):
    """Create a normalized codebook entry while preserving optional code_id."""
    entry = {
        'name': _clean_code_name(name),
        'definition': _clean_optional_text(definition),
        'inclusion_criteria': inclusion_criteria or [],
        'exclusion_criteria': exclusion_criteria or [],
        'example_verbatims': example_verbatims or []
    }
    cid = _clean_code_id(code_id)
    if cid:
        entry[CODE_ID_FIELD] = cid
    entry.update({k: v for k, v in extra.items() if v not in (None, '')})
    return entry


def get_code_columns(df, codebook):
    """
    Return [(name, code_col, conf_col)] for every code whose binary column
    (code_<name>) and confidence column (conf_<name>) both exist in df.

    Code IDs are preserved in the codebook/reporting layer but existing binary
    columns remain name-based to avoid breaking downstream notebook logic.
    """
    return [(c['name'], f"code_{c['name']}", f"conf_{c['name']}")
            for c in codebook['codes']
            if f"code_{c['name']}" in df.columns
            and f"conf_{c['name']}" in df.columns]
def format_codebook_for_prompt(codebook):
    """Render a codebook dict as a numbered text block for an LLM prompt."""
    lines = []
    for i, code in enumerate(codebook['codes'], 1):
        lines.append(f"{i}. **{_code_ref(code)}**")
        if code.get('definition'):         lines.append(f"   Definition: {code['definition']}")
        if code.get('inclusion_criteria'): lines.append(f"   Include if: {'; '.join(code['inclusion_criteria'])}")
        if code.get('exclusion_criteria'): lines.append(f"   Exclude if: {'; '.join(code['exclusion_criteria'])}")
        lines.append('')
    return '\n'.join(lines)
def format_codebook_for_prompt_compact(codebook):
    """
    Render a shorter codebook for high-volume coding calls.

    If a code ID exists, it is shown for traceability, but the model is still
    instructed to return the exact code name so existing code columns stay stable.
    """
    lines = []
    for i, code in enumerate(codebook['codes'], 1):
        ref = _code_ref(code)
        definition = str(code.get('definition', '')).strip()
        if definition:
            lines.append(f"{i}. {ref}: {definition}")
        else:
            lines.append(f"{i}. {ref}")
    return '\n'.join(lines)
def add_codes_column(df, codebook, separator=' | '):
    """
    Rebuild codes / code_ids / num_codes / has_multi_label summary columns.
    Always drops stale versions before recomputing.
    """
    for col in ['codes', 'code_ids', 'num_codes', 'has_multi_label']:
        if col in df.columns: df.drop(columns=[col], inplace=True)
    entries   = get_code_columns(df, codebook)
    code_cols = [e[1] for e in entries]
    names     = [e[0] for e in entries]
    id_by_name = {c['name']: _clean_code_id(c.get(CODE_ID_FIELD, ''))
                  for c in codebook.get('codes', [])}

    df['codes_list'] = df.apply(
        lambda row: [n for n, c in zip(names, code_cols) if row[c] == 1], axis=1)
    df['code_ids_list'] = df['codes_list'].apply(
        lambda codes: [id_by_name.get(n, '') for n in codes if id_by_name.get(n, '')])
    df['codes']           = df['codes_list'].apply(lambda l: separator.join(l))
    df['code_ids']        = df['code_ids_list'].apply(lambda l: separator.join(l))
    df['num_codes']       = df['codes_list'].apply(len)
    df['has_multi_label'] = df['num_codes'] > 1
    df.drop(columns=['codes_list', 'code_ids_list'], inplace=True)
    return df
def display_codebook(codebook):
    """Pretty-print a codebook to stdout."""
    print('\n' + '='*60)
    print(f"CODEBOOK  version={codebook.get('version','--')}  "
          f"codes={len(codebook['codes'])}  "
          f"multi_label={codebook.get('multi_label', True)}  "
          f"has_code_ids={codebook_has_code_ids(codebook)}")
    print('='*60)
    for i, code in enumerate(codebook['codes'], 1):
        print(f"\n[{i}] {_code_ref(code)}")
        if code.get('definition'):          print(f"    Definition : {code['definition']}")
        if code.get('inclusion_criteria'):  print(f"    Include    : {'; '.join(code['inclusion_criteria'])}")
        if code.get('exclusion_criteria'):  print(f"    Exclude    : {'; '.join(code['exclusion_criteria'])}")
        if code.get('example_verbatims'):   print(f"    Example    : {code['example_verbatims'][0][:80]}...")
        if code.get('parent_code'):         print(f"    Parent     : {code['parent_code']}")
        if code.get('parent_code_id'):      print(f"    Parent ID  : {code['parent_code_id']}")
        if code.get('merged_from'):         print(f"    Merged from: {', '.join(code['merged_from'])}")
        if code.get('merged_from_ids'):     print(f"    Merged IDs : {', '.join(code['merged_from_ids'])}")
def save_codebook(codebook, filepath):
    """Persist a codebook dict to a JSON file."""
    with open(filepath, 'w', encoding='utf-8') as f:
        json.dump(codebook, f, indent=2, ensure_ascii=False)
    logger.info(f'Codebook saved → {filepath}')

def load_codebook(filepath):
    """Load a previously saved codebook from a JSON file."""
    with open(filepath, 'r', encoding='utf-8') as f:
        return json.load(f)

def _extract_json(raw):
    """
    Extract a JSON object or array from an LLM response.

    Strategy
    --------
    1. If the response is fenced (```json ... ``` or ``` ... ```), return
       the fenced contents.
    2. Otherwise, find the first '{' or '[' and scan forward with brace/
       bracket balancing (string- and escape-aware) to return the smallest
       complete JSON value. This avoids the greedy brace-regex match that
       can span unrelated braces in prose, and also handles JSON arrays
       (used by translation and batch-coding calls).
    3. Fall through to raw.strip() so json.loads() raises a clear error
       at the caller if nothing JSON-like is present.
    """
    import re as _re
    text = raw.strip()

    fenced = _re.search(r'```(?:json)?\s*\n?(.*?)\n?\s*```', text, _re.DOTALL)
    if fenced:
        return fenced.group(1).strip()

    start = -1
    open_char = ''
    close_char = ''
    for i, ch in enumerate(text):
        if ch in '{[':
            start = i
            open_char = ch
            close_char = '}' if ch == '{' else ']'
            break
    if start == -1:
        return text

    depth = 0
    in_string = False
    escape = False
    for j in range(start, len(text)):
        ch = text[j]
        if in_string:
            if escape:
                escape = False
            elif ch == '\\':
                escape = True
            elif ch == '"':
                in_string = False
            continue
        if ch == '"':
            in_string = True
            continue
        if ch == open_char:
            depth += 1
        elif ch == close_char:
            depth -= 1
            if depth == 0:
                return text[start:j+1].strip()

    # Unbalanced — return what we have and let json.loads raise.
    return text[start:].strip()

def _validate_codebook(codebook):
    """Assert required fields exist in a codebook dict and code IDs are coherent."""
    if 'codes' not in codebook or not isinstance(codebook['codes'], list) \
       or len(codebook['codes']) == 0:
        raise ValueError("Codebook missing or empty 'codes' list.")
    required = {'name','definition','inclusion_criteria',
                'exclusion_criteria','example_verbatims'}
    names = []
    ids = []
    has_ids = codebook_has_code_ids(codebook)
    for i, code in enumerate(codebook['codes']):
        missing = required - code.keys()
        if missing: raise ValueError(f'Code #{i+1} missing fields: {missing}')
        name = _clean_code_name(code.get('name', ''))
        if not name: raise ValueError(f'Code #{i+1} has a blank name.')
        names.append(name)
        if has_ids:
            cid = _clean_code_id(code.get(CODE_ID_FIELD, ''))
            if not cid:
                raise ValueError(
                    f'Code "{name}" is missing code_id. Because this codebook has '
                    'a code ID column, every code must have an ID.'
                )
            code[CODE_ID_FIELD] = cid
            ids.append(cid)
    dup_names = sorted({n for n in names if names.count(n) > 1})
    if dup_names:
        raise ValueError(f'Duplicate code names are not supported because columns are name-based: {dup_names}')
    if has_ids:
        dup_ids = sorted({x for x in ids if ids.count(x) > 1})
        if dup_ids:
            raise ValueError(f'Duplicate code IDs found: {dup_ids}')
def clean_text(text, *, normalize_form='NFC', remove_control_chars=True,
               fix_entities=True, replace_urls_with='__URL__',
               replace_emails_with='__EMAIL__', emoji_handling='remove',
               emoji_token='__EMOJI__', lowercase=False, strip_accents=False):
    """Sanitise a response string: HTML, encoding, URLs, emoji, whitespace."""
    try:
        if pd.isna(text): return ''
    except Exception: pass
    if text is None: return ''
    if not isinstance(text, str): text = str(text)
    text = html.unescape(text)
    text = ftfy.fix_text(text, normalization=normalize_form,
                         remove_control_chars=remove_control_chars,
                         fix_entities=fix_entities)
    text = HTML_TAG_RE.sub(' ', text)
    text = URL_RE.sub(f' {replace_urls_with} ' if replace_urls_with else ' ', text)
    text = EMAIL_RE.sub(f' {replace_emails_with} ' if replace_emails_with else ' ', text)
    if emoji_handling == 'token':    text = EMOJI_RE.sub(f' {emoji_token} ', text)
    elif emoji_handling == 'remove': text = EMOJI_RE.sub(' ', text)
    if lowercase:     text = text.lower()
    if strip_accents:
        import unicodedata as _ud
        text = ''.join(c for c in _ud.normalize('NFD', text)
                       if _ud.category(c) != 'Mn')
    return re.sub(r'\s+', ' ', text).strip()

def is_valid_response(text, min_words=DEFAULT_MIN_WORDS):
    """Return True only if the response is substantive enough to code."""
    if not text: return False
    t = text.strip()
    if not t or t.lower() in EMPTY_RESPONSES: return False
    if re.fullmatch(r'[\W_]+', t): return False
    if t.lower() in {'__url__', '__email__', '__emoji__'}: return False
    return len(t.split()) >= min_words

def enrich_codebook_with_ai(client, codebook, survey_context='',
                             model=None, max_tokens=8000,
                             fill_missing_with_fallback=False,
                             fallback_parent_code='',
                             fallback_merged_from=None):
    """
    Write definitions, criteria, and verbatims for codes with empty definition.
    All undefined codes are enriched in a single API call (not one per code).
    Codes already defined are skipped. Used by Sections 3, 4, and 5.

    Matching is intentionally robust: the model may return either the exact code
    name, a bracketed "[ID] Name" label, or a code_id. Existing manual
    definitions are never overwritten.

    If fill_missing_with_fallback=True, any remaining blank definitions are
    filled with conservative deterministic definitions so split/merge-generated
    codebooks remain complete even when the LLM output is incomplete.
    """
    if model is None: model = CODING_MODEL
    to_enrich = [c for c in codebook['codes'] if not _clean_optional_text(c.get('definition'))]
    if not to_enrich:
        logger.info('All codes have definitions -- skipping enrichment.')
        return codebook
    logger.info(f'Enriching {len(to_enrich)} codes (1 API call)...')
    code_list = '\n'.join(
        f"- name={json.dumps(c.get('name', ''), ensure_ascii=False)}"
        + (f" | code_id={json.dumps(_clean_code_id(c.get(CODE_ID_FIELD, '')), ensure_ascii=False)}"
           if _clean_code_id(c.get(CODE_ID_FIELD, '')) else '')
        for c in to_enrich
    )
    ctx = survey_context or 'Open-ended survey responses.'
    lang_name = _codebook_language_name()
    prompt = (
        'You are a qualitative research expert.\n\n'
        f'CODEBOOK OUTPUT LANGUAGE: {lang_name}.\n'
        f'Write all generated definitions, criteria, and example verbatims in {lang_name}.\n\n'
        f'CONTEXT:\n{ctx}\n\n'
        f'Codes without descriptions:\n{code_list}\n\n'
        f'For each code, generate in {lang_name}:\n'
        '1. A clear definition (1-2 sentences)\n'
        '2. 2-3 inclusion criteria\n'
        '3. 1-2 exclusion criteria\n'
        '4. 2 example verbatims\n\n'
        'Important constraints:\n'
        '- Return the "name" field exactly as provided above.\n'
        '- If a code_id is provided, also return the same "code_id" field.\n'
        '- Do not rename codes.\n\n'
        'Return ONLY valid JSON, with no Markdown:\n'
        '{"codes":[{"name":"<exact name>","code_id":"<id if provided>","definition":"...",'
        '"inclusion_criteria":["..."],"exclusion_criteria":["..."],'
        '"example_verbatims":["...","..."]}]}'
    )
    try:
        resp = chat_with_retry(
            client, model=model,
            messages=[{'role':'system','content':'Qualitative research expert. Return ONLY valid JSON.'},
                      {'role':'user',  'content':prompt}],
            temperature=0.3, max_tokens=max_tokens)
        enriched = json.loads(_extract_json(resp.choices[0].message.content.strip()))
    except json.JSONDecodeError as e:
        logger.error(f'Enrichment parse error: {e}')
        enriched = {'codes': []}
    except Exception as e:
        logger.error(f'Enrichment call failed: {e}')
        enriched = {'codes': []}

    by_name = {_canonical_code_key(c.get('name', '')): c for c in enriched.get('codes', []) if c.get('name')}
    by_id = {
        _clean_code_id(c.get(CODE_ID_FIELD, '') or c.get('code_id', '')): c
        for c in enriched.get('codes', [])
        if _clean_code_id(c.get(CODE_ID_FIELD, '') or c.get('code_id', ''))
    }

    matched = []
    for code in codebook['codes']:
        if _clean_optional_text(code.get('definition')):
            continue
        cid = _clean_code_id(code.get(CODE_ID_FIELD, ''))
        ec = by_id.get(cid) if cid else None
        if ec is None:
            ec = by_name.get(_canonical_code_key(code.get('name', '')))
        if ec is None:
            continue
        definition = _clean_optional_text(ec.get('definition', ''))
        if not definition:
            continue
        code['definition']         = definition
        code['inclusion_criteria'] = ec.get('inclusion_criteria', []) or code.get('inclusion_criteria', []) or []
        code['exclusion_criteria'] = ec.get('exclusion_criteria', []) or code.get('exclusion_criteria', []) or []
        code['example_verbatims']  = ec.get('example_verbatims', []) or code.get('example_verbatims', []) or []
        code['definition_source']  = code.get('definition_source', 'ai_enrichment')
        matched.append(code.get('name', ''))

    still_missing = [c.get('name', '') for c in codebook['codes']
                     if not _clean_optional_text(c.get('definition'))]
    if still_missing:
        logger.warning('AI enrichment did not fill definitions for: %s', still_missing)
        if fill_missing_with_fallback:
            codebook = _apply_definition_fallbacks(
                codebook,
                survey_context=survey_context,
                parent_code=fallback_parent_code,
                merged_from=fallback_merged_from,
                reason='deterministic_fallback_after_ai_enrichment'
            )
    logger.info('Enrichment complete. Matched definitions for: %s', matched)
    return codebook


### 0-F · Reporting engine
*Called at the end of every section. Never modified between sections.*

| Function | Output |
|----------|--------|
| `calculate_frequencies` | DataFrame: code, count, % |
| `generate_cooccurrence_matrix` | Square DataFrame |
| `plot_frequency_chart` | Horizontal bar chart (.png) |
| `plot_cooccurrence_heatmap` | Annotated heatmap (.png) |
| `generate_reports` | All of the above + 5-sheet Excel workbook |


In [105]:
def calculate_frequencies(df, codebook):
    """
    Count how many responses carry each code.
    Returns a DataFrame sorted descending by count, with code_id preserved when available.
    """
    entries = get_code_columns(df, codebook)
    cols = ['code_id', 'code', 'count', 'percentage'] if codebook_has_code_ids(codebook) else ['code', 'count', 'percentage']
    if not entries: return pd.DataFrame(columns=cols)
    total = len(df)
    id_by_name = {c['name']: _clean_code_id(c.get(CODE_ID_FIELD, ''))
                  for c in codebook.get('codes', [])}
    rows = []
    for n, cc, _ in entries:
        row = {'code': n,
               'count': int(df[cc].sum()),
               'percentage': round(int(df[cc].sum())/total*100,1) if total else 0}
        if codebook_has_code_ids(codebook):
            row['code_id'] = id_by_name.get(n, '')
        rows.append(row)
    return pd.DataFrame(rows)[cols].sort_values('count', ascending=False).reset_index(drop=True)
def generate_cooccurrence_matrix(df, codebook):
    """
    Build a code × code co-occurrence matrix.
    Cell (i,j) = responses assigned both code i and code j.
    Diagonal  = individual code frequency.
    """
    entries   = get_code_columns(df, codebook)
    code_cols = [e[1] for e in entries]
    names     = [e[0] for e in entries]
    if not code_cols: return pd.DataFrame()
    binary = df[code_cols].values.astype(int)
    return pd.DataFrame(binary.T @ binary, index=names, columns=names)

def plot_frequency_chart(freq_df, output_dir, title='Fréquence des codes', figsize=(12,7)):
    """
    Save a horizontal bar chart of code frequencies.
    Bars are sorted top-to-bottom descending. Percentage labels appear on the right.
    """
    output_dir = Path(output_dir)
    plot_df = freq_df.sort_values('count', ascending=True)
    fig, ax = plt.subplots(figsize=figsize)
    colors  = sns.color_palette('viridis', len(plot_df))
    bars    = ax.barh(plot_df['code'], plot_df['count'], color=colors)
    ax.set_xlabel('Nombre de réponses'); ax.set_title(title)
    for bar, pct in zip(bars, plot_df['percentage']):
        ax.text(bar.get_width()+0.3, bar.get_y()+bar.get_height()/2,
                f'{pct}%', va='center', fontsize=9)
    plt.tight_layout()
    fp = output_dir / f'{FILE_STEM}_frequencies.png'
    plt.savefig(fp, dpi=150, bbox_inches='tight'); plt.close()
    logger.info(f'Frequency chart → {fp}'); return str(fp)

def plot_cooccurrence_heatmap(cooc_df, output_dir,
                               title='Co-occurrence des codes', figsize=(12,10)):
    """
    Save a co-occurrence heatmap.  The diagonal is masked so the colour scale
    reflects off-diagonal values rather than individual code frequencies.
    """
    output_dir = Path(output_dir)
    if cooc_df.empty:
        logger.warning('Co-occurrence matrix empty — skipping heatmap.'); return None
    fig, ax = plt.subplots(figsize=figsize)
    sns.heatmap(cooc_df, mask=np.eye(len(cooc_df), dtype=bool),
                annot=True, fmt='d', cmap='YlOrRd', square=True, linewidths=0.5,
                cbar_kws={'label':'Co-occurrences'}, ax=ax)
    ax.set_title(title); plt.xticks(rotation=45, ha='right'); plt.tight_layout()
    fp = output_dir / f'{FILE_STEM}_cooccurrence.png'
    plt.savefig(fp, dpi=150, bbox_inches='tight'); plt.close()
    logger.info(f'Co-occurrence heatmap → {fp}'); return str(fp)

def generate_reports(df, codebook, output_dir,
                     text_column=None, output_file='coding_summary.xlsx',
                     include_charts=True, include_excel=True):
    """
    Generate the full set of analysis outputs for a coded dataset.

    Excel sheets produced:
      Résumé          — high-level counts (total, valid, coded, uncoded)
      Fréquences      — code frequency table with percentages
      Co-occurrence   — code co-occurrence matrix
      Codebook        — definitions, criteria, audit trail (parent / merged_from)
      Données codées  — one row per response with all code and confidence columns

    Parameters
    ----------
    text_column : defaults to the global TEXT_COL constant
    """
    if text_column is None: text_column = TEXT_COL
    output_dir = Path(output_dir); output_dir.mkdir(parents=True, exist_ok=True)
    freq_df = calculate_frequencies(df, codebook)
    cooc_df = generate_cooccurrence_matrix(df, codebook)
    outputs = {}
    if include_charts:
        outputs['frequency_chart']    = plot_frequency_chart(freq_df, output_dir)
        outputs['cooccurrence_chart'] = plot_cooccurrence_heatmap(cooc_df, output_dir)
    if include_excel:
        path  = output_dir / output_file
        total = len(df)
        nv    = int(df['is_valid'].sum()) if 'is_valid' in df.columns else total
        ccols = [e[1] for e in get_code_columns(df, codebook)]
        nc    = int((df[ccols].sum(axis=1) > 0).sum()) if ccols else 0
        summ  = pd.DataFrame({'Métrique':['Total réponses','Réponses valides',
                    'Invalides/vides','Réponses codées','Non codées','Nb codes','Multi-label'],
                    'Valeur':[total,nv,total-nv,nc,total-nc,
                              len(codebook['codes']),codebook.get('multi_label',True)]})
        cb_rows = []
        for c in codebook['codes']:
            row = {
                'Code ID': _clean_code_id(c.get(CODE_ID_FIELD, '')),
                'Code': c['name'],
                'Définition': c.get('definition',''),
                'Inclusion': '; '.join(c.get('inclusion_criteria',[])),
                'Exclusion': '; '.join(c.get('exclusion_criteria',[])),
                'Exemples':  ' | '.join(c.get('example_verbatims',[])[:3]),
                'Parent':    c.get('parent_code',''),
                'Parent Code ID': c.get('parent_code_id',''),
                'Fusionné de': ' | '.join(c.get('merged_from',[])),
                'Fusionné de IDs': ' | '.join(c.get('merged_from_ids',[])),
                'Source': c.get('source','')
            }
            if not codebook_has_code_ids(codebook):
                row.pop('Code ID', None)
                row.pop('Parent Code ID', None)
                row.pop('Fusionné de IDs', None)
            cb_rows.append(row)
        cbdf = pd.DataFrame(cb_rows)
        base  = [c for c in ['response_id','original_text','codes','code_ids','has_multi_label',
                              'num_codes',text_column,'is_valid'] if c in df.columns]
        ccol2 = [c for c in df.columns if c.startswith('code_')]
        with pd.ExcelWriter(path, engine='openpyxl') as w:
            summ.to_excel(w,  sheet_name='Résumé',        index=False)
            freq_df.to_excel(w, sheet_name='Fréquences',  index=False)
            cooc_df.to_excel(w, sheet_name='Co-occurrence')
            cbdf.to_excel(w,  sheet_name='Codebook',      index=False)
            df[base+ccol2].to_excel(w, sheet_name='Données codées', index=False)
        outputs['summary_report'] = str(path)
        logger.info(f'Report → {path}')
    return outputs


### 0-G · Pipeline state & checkpoints
*Reset and restore machinery lives here so Section 4 and Section 5 configuration cells stay focused.*

| Helper | Purpose |
|--------|---------|
| `set_pipeline_baseline(name, df, codebook)` | Save a complete dataframe + codebook state in memory and, by default, on disk |
| `reset_pipeline_state(name)` | Restore `df_current` and `codebook_current` from a named checkpoint |
| `reset_to_section2()` / `reset_to_section3()` | Return to the clean Section 2 or Section 3 output before split/merge experiments |
| `list_pipeline_checkpoints()` | Show available memory and disk checkpoints |

Recommended checkpoint flow: run Section 2 or Section 3 first, then use Section 4/5 for split and merge experiments. 
Checkpoint names used by this notebook include `s2`, `s3`, `pre_s4`, `post_s4`, `pre_s5`, and `post_s5`.

### 0-G-1 · Pipeline checkpoint functions

In [106]:
# ── Pipeline checkpoint + reset helpers ───────────────────────────────────
# Robust reset means restoring BOTH the coded dataframe and the matching codebook.
# The helpers below create in-memory baselines for fast reset and disk checkpoints
# for recovery after a notebook/kernel restart.
PIPELINE_BASELINES = {}

def _checkpoint_key(name):
    """Normalize checkpoint names such as 'Section 3', 's3', 'pre_s4'."""
    key = str(name).strip().lower().replace('section ', 's').replace(' ', '_').replace('-', '_')
    aliases = {
        '2': 's2', 'section2': 's2', 'section_2': 's2',
        '3': 's3', 'section3': 's3', 'section_3': 's3',
        '4': 's4', 'section4': 's4', 'section_4': 's4',
        '5': 's5', 'section5': 's5', 'section_5': 's5',
    }
    return aliases.get(key, key)

def _checkpoint_root(checkpoint_dir=None):
    """Return the checkpoint directory as a Path and create it when needed."""
    root = Path(checkpoint_dir or globals().get('CHECKPOINT_DIR', 'checkpoints'))
    root.mkdir(parents=True, exist_ok=True)
    return root

def _checkpoint_paths(name, checkpoint_dir=None):
    """Return standardized checkpoint paths for a named pipeline state."""
    key = _checkpoint_key(name)
    root = _checkpoint_root(checkpoint_dir)
    return {
        'key': key,
        'df': root / f'{key}__df.pkl',
        'codebook': root / f'{key}__codebook.json',
        'metadata': root / f'{key}__metadata.json',
    }

def _clone_pipeline_state(df, codebook):
    """Deep-copy the dataframe and codebook so later split/merge edits cannot mutate the baseline."""
    return df.copy(deep=True), copy.deepcopy(codebook)

def save_pipeline_checkpoint(name, df, codebook, checkpoint_dir=None, metadata=None):
    """
    Save a complete pipeline checkpoint to disk.

    A complete checkpoint includes:
    - the coded dataframe, including code_* columns and summary columns
    - the matching codebook JSON
    - metadata explaining where the checkpoint came from
    """
    paths = _checkpoint_paths(name, checkpoint_dir)
    df_to_save, codebook_to_save = _clone_pipeline_state(df, codebook)
    df_to_save.to_pickle(paths['df'])
    save_codebook(codebook_to_save, paths['codebook'])

    meta = {
        'checkpoint_key': paths['key'],
        'created_at_utc': datetime.now(timezone.utc).isoformat(),
        'file_stem': globals().get('FILE_STEM', ''),
        'rows': int(len(df_to_save)),
        'columns': list(df_to_save.columns),
        'code_count': int(len(codebook_to_save.get('codes', []))),
    }
    if metadata:
        meta.update(metadata)
    with open(paths['metadata'], 'w', encoding='utf-8') as f:
        json.dump(meta, f, indent=2, ensure_ascii=False, default=str)

    logger.info(f'Pipeline checkpoint saved → {paths["key"]} ({paths["df"].parent})')
    return paths

def set_pipeline_baseline(name, df, codebook, checkpoint_dir=None, metadata=None, save_to_disk=None):
    """
    Store a reset baseline in memory and, by default, on disk.

    Use this after Section 2 or Section 3 completes. It defines the clean starting
    point for later Section 4 split and Section 5 merge experiments.
    """
    key = _checkpoint_key(name)
    df_copy, codebook_copy = _clone_pipeline_state(df, codebook)
    PIPELINE_BASELINES[key] = {
        'df': df_copy,
        'codebook': codebook_copy,
        'metadata': metadata or {},
        'created_at_utc': datetime.now(timezone.utc).isoformat(),
    }

    # Also expose convenient variables such as df_baseline_s3 and codebook_baseline_s3.
    globals()[f'df_baseline_{key}'] = df_copy.copy(deep=True)
    globals()[f'codebook_baseline_{key}'] = copy.deepcopy(codebook_copy)

    if save_to_disk is None:
        save_to_disk = bool(globals().get('SAVE_RESET_CHECKPOINTS_TO_DISK', True))
    if save_to_disk:
        save_pipeline_checkpoint(key, df_copy, codebook_copy,
                                 checkpoint_dir=checkpoint_dir, metadata=metadata)

    print(f'✅ Reset baseline set: {key} — {len(df_copy):,} rows, {len(codebook_copy.get("codes", []))} codes')
    return df_copy, codebook_copy

def load_pipeline_checkpoint(name, checkpoint_dir=None, set_current=True, rebuild_summary=True):
    """
    Load a complete dataframe + codebook checkpoint from disk.

    This is safer than loading only the codebook, because the dataframe and codebook
    must match exactly after split/merge operations.
    """
    global df_current, codebook_current
    paths = _checkpoint_paths(name, checkpoint_dir)
    missing = [str(p) for p in [paths['df'], paths['codebook']] if not p.exists()]
    if missing:
        raise FileNotFoundError(
            'Checkpoint is incomplete or missing. Run Section 2/3 first, or save the checkpoint before loading it.\n'
            f'Missing: {missing}'
        )

    df_loaded = pd.read_pickle(paths['df'])
    codebook_loaded = load_codebook(paths['codebook'])
    _validate_codebook(codebook_loaded)

    if rebuild_summary:
        df_loaded = add_codes_column(df_loaded.copy(deep=True), codebook_loaded)

    if set_current:
        df_current, codebook_current = _clone_pipeline_state(df_loaded, codebook_loaded)
        print(f'✅ Loaded checkpoint into df_current/codebook_current: {paths["key"]}')
        return df_current, codebook_current
    return df_loaded, codebook_loaded

def reset_pipeline_state(name='s3', prefer='auto', checkpoint_dir=None, rebuild_summary=True):
    """
    Restore df_current and codebook_current to a named checkpoint.

    Parameters
    ----------
    name : str
        Examples: 's2', 's3', 'pre_s4', 'post_s4', 'pre_s5', 'post_s5'.
    prefer : {'auto', 'memory', 'disk'}
        auto   = use in-memory baseline when available, otherwise disk checkpoint.
        memory = require in-memory baseline.
        disk   = require disk checkpoint.
    """
    global df_current, codebook_current
    key = _checkpoint_key(name)
    prefer = str(prefer).lower().strip()
    if prefer not in {'auto', 'memory', 'disk'}:
        raise ValueError("prefer must be one of: 'auto', 'memory', 'disk'")

    if prefer in {'auto', 'memory'} and key in PIPELINE_BASELINES:
        df_current, codebook_current = _clone_pipeline_state(
            PIPELINE_BASELINES[key]['df'], PIPELINE_BASELINES[key]['codebook']
        )
        if rebuild_summary:
            df_current = add_codes_column(df_current.copy(deep=True), codebook_current)
        print(f'✅ Reset complete from memory baseline: {key}')
        return df_current, codebook_current

    if prefer == 'memory':
        raise KeyError(
            f'No in-memory baseline named {key!r}. Use prefer="auto" or prefer="disk", '
            'or rerun the section that creates this baseline.'
        )

    return load_pipeline_checkpoint(key, checkpoint_dir=checkpoint_dir,
                                    set_current=True, rebuild_summary=rebuild_summary)

def reset_to_section2(prefer='auto'):
    """Restore the clean Section 2 output before any Section 4/5 refinements."""
    return reset_pipeline_state('s2', prefer=prefer)

def reset_to_section3(prefer='auto'):
    """Restore the clean Section 3 output before any Section 4/5 refinements."""
    return reset_pipeline_state('s3', prefer=prefer)

def list_pipeline_checkpoints(checkpoint_dir=None):
    """List in-memory and on-disk checkpoints available for reset."""
    root = _checkpoint_root(checkpoint_dir)
    rows = []

    for key, item in sorted(PIPELINE_BASELINES.items()):
        rows.append({
            'checkpoint': key,
            'source': 'memory',
            'created_at_utc': item.get('created_at_utc', ''),
            'rows': len(item['df']),
            'codes': len(item['codebook'].get('codes', [])),
            'description': item.get('metadata', {}).get('description', ''),
        })

    for meta_path in sorted(root.glob('*__metadata.json')):
        try:
            with open(meta_path, 'r', encoding='utf-8') as f:
                meta = json.load(f)
        except Exception:
            meta = {}
        rows.append({
            'checkpoint': meta.get('checkpoint_key', meta_path.name.replace('__metadata.json', '')),
            'source': 'disk',
            'created_at_utc': meta.get('created_at_utc', ''),
            'rows': meta.get('rows', ''),
            'codes': meta.get('code_count', ''),
            'description': meta.get('description', ''),
        })

    if not rows:
        print('No checkpoints found yet. Run Section 2 or Section 3 to create a reset baseline.')
        return pd.DataFrame(columns=['checkpoint','source','created_at_utc','rows','codes','description'])
    return pd.DataFrame(rows).drop_duplicates(subset=['checkpoint','source']).reset_index(drop=True)


### 0-G-2 · Reset command examples

In [107]:
# ── Reset examples ───────────────────────────────────────────────────────
# View all memory and disk checkpoints.
# list_pipeline_checkpoints()

# Restore clean Section 3 output before trying a new split/merge strategy.
# df_current, codebook_current = reset_to_section3()

# Restore clean Section 2 output instead.
# df_current, codebook_current = reset_to_section2()

# Undo the latest split.
# df_current, codebook_current = reset_pipeline_state('pre_s4')

# Restore the latest post-split state before testing merge options.
# df_current, codebook_current = reset_pipeline_state('post_s4')

# Undo the latest merge.
# df_current, codebook_current = reset_pipeline_state('pre_s5')

# Restore after a kernel restart using disk checkpoints.
# First rerun Section 0 so the reset functions are defined, then run one of:
# df_current, codebook_current = reset_pipeline_state('s3', prefer='disk')
# df_current, codebook_current = reset_pipeline_state('post_s4', prefer='disk')

# ── Example: split → multi-merge chain ────────────────────────────────────────
# Step 1 — start from Section 3 output
# df_current, codebook_current = reset_to_section3()
#
# Step 2 — split one or more broad codes
# df_current, codebook_current, _ = split_codes(
#     df=df_current, codebook=codebook_current, client=client,
#     split_configs=S4_SPLIT_CONFIGS,
#     output_dir='reports_chain/')
#
# Step 3 — merge one or more groups of similar codes in the updated result
# df_current, codebook_current, _ = merge_code_groups(
#     df=df_current, codebook=codebook_current, client=client,
#     merge_configs=[
#         {'codes_to_merge': ['parent code1', 'parent code2'], 'merged_name': 'merge1'},
#         {'codes_to_merge': ['parent code5', 'parent code7', 'parent code8'], 'merged_name': 'merge2'},
#     ],
#     output_dir='reports_chain/')
#
# Step 4 — save the final state
# save_codebook(codebook_current, FILE_STEM + '_codebook_final.json')
# df_current.to_csv(FILE_STEM + '_final.csv', index=False)

print('Pipeline checkpoint helpers are ready. Use these examples by uncommenting only the command you need.')


Pipeline checkpoint helpers are ready. Use these examples by uncommenting only the command you need.


---
## SECTION 1 · Preprocessing
*Run once per dataset. Produces a clean, translated DataFrame ready for Sections 2 or 3.*

### Processing steps
```
Raw file (.xlsx / .csv)
  ↓  load_survey_data()          — read file, assign response_id, drop blank rows
  ↓  preprocess_data()           — clean text, remove invalid & duplicate responses
                                   saves removed_records.xlsx for audit
  ↓  translate_source_records() — translate selected detected source language into target language
                                   default: EN → FR; skipped if no matching source responses
  ↓
df_preprocessed  (TEXT_COL = "final_text_fr")
```

### Columns added by Section 1
| Column | Description |
|--------|-------------|
| `response_id` | Stable integer ID (1-based) |
| `original_text` | Verbatim as loaded from file |
| `cleaned_text` | After `clean_text()` |
| `detected_lang` | ISO language code detected by Lingua, e.g., `en`, `fr`, `es`, or `other` |
| `final_text_<target>` | Target-language text — default `final_text_fr`, translated when source is detected |


### 1-A · Preprocessing functions

In [108]:
def load_survey_data(filepath, text_column, sheet_name=0, id_column=None):
    """Load survey responses from Excel or CSV. Drops blank rows."""
    path = Path(filepath)
    df   = (pd.read_excel(path, sheet_name=sheet_name)
            if path.suffix.lower() in ('.xlsx','.xls') else pd.read_csv(path))
    if text_column not in df.columns:
        raise ValueError(f"Column '{text_column}' not found. "
                         f"Available: {list(df.columns)}")
    initial = len(df)
    df = df[df[text_column].notna() &
            df[text_column].astype(str).str.strip().ne('')].copy()
    df['response_id']   = (df[id_column] if id_column and id_column in df.columns
                           else range(1, len(df)+1))
    df['original_text'] = df[text_column].astype(str)
    logger.info(f'Loaded {len(df)} responses ({initial-len(df)} blank rows removed)')
    return df

# ── Lingua language detection ────────────────────────────────────────────────
# Keep this list focused on languages you expect in the survey.
# Add/remove languages here if your data changes.
LINGUA_LANGUAGES = [
    Language.ENGLISH,
    Language.FRENCH,
    Language.SPANISH,
    Language.GERMAN,
    Language.ITALIAN,
    Language.PORTUGUESE,
    Language.DUTCH,
    Language.ARABIC,
]

LINGUA_DETECTOR = LanguageDetectorBuilder.from_languages(*LINGUA_LANGUAGES).build()


def detect_language_lingua(text):
    """
    Detect language with Lingua and return a lowercase ISO-639-1 code.

    Examples: 'en', 'fr', 'es'. Returns 'other' when Lingua cannot detect.
    """
    if not text or not str(text).strip():
        return 'other'

    lang = LINGUA_DETECTOR.detect_language_of(str(text))
    if lang is None:
        return 'other'

    # lingua exposes ISO code enums such as ISOCode639_1.EN.
    return lang.iso_code_639_1.name.lower()


def preprocess_data(df, min_words=DEFAULT_MIN_WORDS,
                    save_invalid_path='removed_records.xlsx'):
    """Clean, validate, de-duplicate, and detect language with Lingua."""
    logger.info('Starting preprocessing...')
    df['cleaned_text'] = df['original_text'].apply(clean_text)
    initial = len(df)
    df['is_valid']     = df['cleaned_text'].apply(
        lambda x: is_valid_response(x, min_words))
    df['is_duplicate'] = df.duplicated(subset=['cleaned_text'], keep='first')
    inv = df[~df['is_valid']].copy()
    dup = df[df['is_duplicate']].copy()
    if len(inv) + len(dup) > 0:
        with pd.ExcelWriter(save_invalid_path, engine='openpyxl') as w:
            inv.to_excel(w, sheet_name='Invalid Records',   index=False)
            dup.to_excel(w, sheet_name='Duplicate Records', index=False)
        logger.info(f'Audit file saved -> {save_invalid_path}')
    df = df[df['is_valid'] & ~df['is_duplicate']].copy()
    df.drop(columns=['is_valid','is_duplicate'], inplace=True)
    logger.info(f'Removed {initial-len(df)} records. {len(df)} remaining.')
    df['detected_lang'] = df['cleaned_text'].apply(detect_language_lingua)
    return df

# Errors that must never be swallowed by translation fallback logic.
# RuntimeError is raised by chat_with_retry for quota and auth failures.
_FATAL_ERRORS = (RuntimeError, KeyboardInterrupt, SystemExit)


def _translate_batch(client, texts, source_language_name=None, target_language_name=None,
                     model=None):
    """
    Translate a list of strings from the selected source language to target language
    in ONE API call.

    Default from Section 0-B:
      SOURCE_LANGUAGE_CODE = 'en'  → English
      TARGET_LANGUAGE_CODE = 'fr'  → French
    """
    if model is None:
        model = CODING_MODEL
    if source_language_name is None:
        source_language_name = SOURCE_LANGUAGE_NAME
    if target_language_name is None:
        target_language_name = TARGET_LANGUAGE_NAME

    numbered = '\n'.join(f'[{i+1}] {t}' for i, t in enumerate(texts))
    prompt = (
        f'Translate the following {len(texts)} survey responses '
        f'from {source_language_name.upper()} to {target_language_name.upper()}.\n\n'
        f'{numbered}\n\n'
        f'Return ONLY a JSON array with exactly {len(texts)} strings, '
        f'preserving order. Example: ["translation 1", "translation 2"]'
    )
    resp = chat_with_retry(
        client, model=model,
        messages=[{'role':'system',
                   'content':f'Translate to {target_language_name}. Return only a JSON array.'},
                  {'role':'user', 'content':prompt}],
        temperature=0.1, max_tokens=min(4000, len(texts) * 120))
    raw = re.sub(r'^```(?:json)?\n?', '',
                 resp.choices[0].message.content.strip())
    raw = re.sub(r'\n?```$', '', raw)
    result = json.loads(raw)
    if not isinstance(result, list) or len(result) != len(texts):
        raise ValueError(f'Expected {len(texts)} translations, got {len(result)}')
    return [str(t).strip() for t in result]


def translate_source_records(df, client,
                             source_lang_code=None,
                             target_lang_code=None,
                             text_column='cleaned_text',
                             output_column=None,
                             batch_size=None, delay=0.3):
    """
    Translate records whose detected language equals source_lang_code into target_lang_code.

    Defaults:
      source_lang_code = SOURCE_LANGUAGE_CODE = 'en'
      target_lang_code = TARGET_LANGUAGE_CODE = 'fr'
      output_column    = TEXT_COL = 'final_text_fr'

    To let the user choose a different target, change TARGET_LANGUAGE_CODE in Section 0-B
    before running preprocessing. Example: TARGET_LANGUAGE_CODE = 'es'.
    """
    if source_lang_code is None:
        source_lang_code = SOURCE_LANGUAGE_CODE
    if target_lang_code is None:
        target_lang_code = TARGET_LANGUAGE_CODE
    if output_column is None:
        output_column = TEXT_COL
    if batch_size is None:
        batch_size = TRANSLATION_BATCH_SIZE

    source_name = LANGUAGE_NAME_BY_CODE.get(source_lang_code, source_lang_code)
    target_name = LANGUAGE_NAME_BY_CODE.get(target_lang_code, target_lang_code)

    df[output_column] = df[text_column]

    # Nothing to translate when source and target are the same.
    if source_lang_code == target_lang_code:
        logger.info('Source and target language are the same. Translation skipped.')
        return df

    source_idx = df[df['detected_lang'] == source_lang_code].index.tolist()
    if not source_idx:
        logger.info(f'No {source_name} responses to translate.'); return df

    n = len(source_idx)
    n_calls = -(-n // batch_size)   # ceiling division
    logger.info(f'Translating {n} {source_name} responses to {target_name}: '
                f'{n_calls} API call(s) of {batch_size} texts each')

    for i in range(0, n, batch_size):
        batch_idx   = source_idx[i:i+batch_size]
        batch_texts = df.loc[batch_idx, text_column].tolist()
        try:
            translated = _translate_batch(
                client, batch_texts,
                source_language_name=source_name,
                target_language_name=target_name
            )
            for idx, t in zip(batch_idx, translated):
                df.at[idx, output_column] = t
            logger.info(f'Translated {min(i+batch_size, n)}/{n}')

        except _FATAL_ERRORS:
            raise

        except Exception as e:
            logger.warning(f'Batch parse failed ({type(e).__name__}: {e}); '
                           f'retrying {len(batch_idx)} responses one-by-one...')
            for idx in batch_idx:
                try:
                    r = _translate_batch(
                        client, [df.at[idx, text_column]],
                        source_language_name=source_name,
                        target_language_name=target_name
                    )
                    df.at[idx, output_column] = r[0]
                except _FATAL_ERRORS:
                    raise
                except Exception as e2:
                    logger.error(f'Translation failed for response ID {idx} '
                                 f'({type(e2).__name__}: {e2}). '
                                 f'Original text kept as-is.')
        time.sleep(delay)

    logger.info('Translation complete.')
    return df


# Backward-compatible alias for older notebook cells/comments.
def translate_english_records(df, client, text_column='cleaned_text', batch_size=None, delay=0.3):
    """Backward-compatible wrapper: translate English records into the selected target language."""
    return translate_source_records(
        df, client,
        source_lang_code='en',
        target_lang_code=TARGET_LANGUAGE_CODE,
        text_column=text_column,
        output_column=TEXT_COL,
        batch_size=batch_size,
        delay=delay,
    )


### 1-B · Run preprocessing

In [109]:
if not OPENAI_API_KEY:
    raise RuntimeError(
        "OPENAI_API_KEY is not set. Set it outside the notebook before running API cells.\n"
        "Windows PowerShell:  $env:OPENAI_API_KEY = 'sk-...'\n"
        "macOS/Linux bash:    export OPENAI_API_KEY='sk-...'"
    )

client = OpenAI(api_key=OPENAI_API_KEY, max_retries=0, timeout=OPENAI_TIMEOUT_SECONDS)

# Load raw data
df_raw = load_survey_data(SURVEY_FILE, SURVEY_COL, SURVEY_SHEET)

# Clean, validate, de-duplicate, and detect language
df_preprocessed = preprocess_data(df_raw)

# Translate selected detected source language into selected target language
# Default: detected source-language responses → selected target language
df_preprocessed = translate_source_records(
    df_preprocessed,
    client,
    source_lang_code=SOURCE_LANGUAGE_CODE,
    target_lang_code=TARGET_LANGUAGE_CODE,
    output_column=TEXT_COL,
)

# Persist for reuse across sessions
df_preprocessed.to_csv(FILE_STEM + "_processed.csv", index=False)

print(f"\n✅ Preprocessing complete — {len(df_preprocessed)} responses ready")
df_preprocessed[["response_id","original_text","detected_lang",TEXT_COL]].head(5)


01:11:06 - INFO - Loaded 41 responses (2 blank rows removed)
01:11:06 - INFO - Starting preprocessing...
01:11:06 - INFO - Audit file saved -> removed_records.xlsx
01:11:06 - INFO - Removed 1 records. 40 remaining.
01:11:07 - INFO - Translating 2 English responses to French: 1 API call(s) of 10 texts each
01:11:08 - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
01:11:08 - INFO - LLM usage | model=gpt-4.1-mini | prompt=84 | completion=22 | total=106 | cached=0
01:11:08 - INFO - Translated 2/2
01:11:09 - INFO - Translation complete.



✅ Preprocessing complete — 40 responses ready


,response_id,original_text,detected_lang,final_text_fr
0,1,Parce que son goût est bien sucré et c'est raf...,fr,Parce que son goût est bien sucré et c'est raf...
1,2,It taste good and it's very refresshing,en,Ça a bon goût et c'est très rafraîchissant
2,3,Parce que j'aime son goût de grenadine et c'es...,fr,Parce que j'aime son goût de grenadine et c'es...
3,4,The taste is so sweet,en,Le goût est tellement sucré
4,5,"Parce que j'aime son goût de grenadine, son go...",fr,"Parce que j'aime son goût de grenadine, son go..."


---
## SECTION 2 · AI-Generated Codebook
*Use when you have no existing codebook and want the AI to discover themes.*  
*Prerequisite: Section 1 complete, `df_preprocessed` exists.*

### Processing pipeline
```
df_preprocessed
  ↓  generate_embeddings()         — OpenAI text-embedding-3-small (1536-dim)
  ↓  reduce_dimensions_umap()      — compress to ~10 dims (cosine metric)
  ↓  cluster_responses()           — HDBSCAN density clustering
  ↓  get_representative_samples()  — N verbatims per cluster shown to configured LLM
  ↓  generate_codebook()           — configured LLM writes professional codebook
  ↓  apply_codes_to_dataframe()    — configured LLM codes every response
  ↓  generate_reports()            — Excel + charts
  ↓
(df_s2, codebook_s2)  →  assign to df_current, codebook_current for Sections 4/5
```

### Tuning clustering parameters (Section 0-B)
| Parameter | Effect |
|-----------|--------|
| Increase `CLUSTER_UMAP_COMPONENTS` | More theme separation, slower |
| Increase `CLUSTER_UMAP_NEIGHBORS` | Broader clusters |
| Decrease `CLUSTER_EPSILON` | More, smaller clusters |
| Decrease `CLUSTER_MIN_SAMPLES` | More noise points promoted to clusters |


### 2-A · Theme discovery functions

In [110]:
from sklearn.metrics.pairwise import cosine_distances
import hdbscan, umap

def _get_openai_client(client_arg=None):
    """
    Return an OpenAI client for embeddings.

    Priority:
      1. Explicit client passed to generate_embeddings()
      2. Existing global client created in Section 1
      3. New client created from OPENAI_API_KEY
    """
    if client_arg is not None:
        return client_arg

    existing_client = globals().get("client")
    if existing_client is not None:
        return existing_client

    if not OPENAI_API_KEY:
        raise RuntimeError("OPENAI_API_KEY is not set. Cannot generate OpenAI embeddings.")

    return OpenAI(api_key=OPENAI_API_KEY, max_retries=0, timeout=globals().get('OPENAI_TIMEOUT_SECONDS', 180.0))

def generate_embeddings(texts, client=None, model=None, batch_size=None):
    """
    Encode texts into dense vectors using OpenAI's small embedding model.

    Default model:
      EMBEDDING_MODEL = "text-embedding-3-small"

    Notes:
      - OpenAI text-embedding-3-small returns 1536-dimensional vectors by default.
      - Embeddings are generated in batches to reduce request overhead.
      - The returned NumPy array remains compatible with UMAP + HDBSCAN below.
    """
    if model is None:
        model = EMBEDDING_MODEL
    if batch_size is None:
        batch_size = EMBEDDING_BATCH_SIZE

    openai_client = _get_openai_client(client)

    # Normalize input for the API
    texts = ["" if pd.isna(t) else str(t) for t in texts]

    all_vectors = []
    total_tokens = 0

    for start in tqdm(range(0, len(texts), batch_size), desc=f"Embedding with {model}"):
        batch = texts[start:start + batch_size]

        resp = openai_client.embeddings.create(
            model=model,
            input=batch,
            timeout=globals().get('OPENAI_TIMEOUT_SECONDS', 180.0)
        )

        # The API returns one embedding per input, indexed in order.
        batch_vectors = [item.embedding for item in sorted(resp.data, key=lambda x: x.index)]
        all_vectors.extend(batch_vectors)

        usage = getattr(resp, "usage", None)
        if usage is not None:
            total_tokens += getattr(usage, "total_tokens", 0) or 0

    emb = np.asarray(all_vectors, dtype=np.float32)

    logger.info(
        "Embeddings: %s | model=%s | input_texts=%s | tokens=%s",
        emb.shape, model, len(texts), total_tokens
    )

    return emb

def reduce_dimensions_umap(embeddings, n_components=None, n_neighbors=None):
    """
    Compress high-dimensional embeddings with UMAP before clustering.
    Uses cosine metric in embedding space; HDBSCAN then works on euclidean.
    n_components is capped at (n_samples - 2) and embedding_dim to avoid errors.
    """
    if n_components is None: n_components = CLUSTER_UMAP_COMPONENTS
    if n_neighbors  is None: n_neighbors  = CLUSTER_UMAP_NEIGHBORS
    n_components = min(n_components, len(embeddings)-2, embeddings.shape[1])
    n_neighbors  = min(n_neighbors,  len(embeddings)-1)
    reducer = umap.UMAP(n_components=n_components, n_neighbors=n_neighbors,
                        min_dist=0.0, metric='cosine', random_state=42,
                        n_jobs=1, low_memory=True)
    reduced = reducer.fit_transform(embeddings)
    logger.info(f'UMAP: {embeddings.shape[1]}D → {reduced.shape[1]}D'); return reduced

def cluster_responses(embeddings, min_cluster_size=5,
                      min_samples=None, epsilon=None, metric='euclidean'):
    """
    Cluster embeddings with HDBSCAN.

    Two paths based on metric:
      'euclidean'  — pass UMAP-reduced embeddings directly (recommended)
      'cosine'     — pre-compute distance matrix (O(n²); avoid above ~5 000 rows)

    Noise points are labelled -1 and excluded from codebook generation.
    """
    if min_samples is None: min_samples = CLUSTER_MIN_SAMPLES
    if epsilon     is None: epsilon     = CLUSTER_EPSILON
    fit_input, hdb_m = ((cosine_distances(embeddings), 'precomputed')
                        if metric == 'cosine' else (embeddings, metric))
    clusterer = hdbscan.HDBSCAN(
        min_cluster_size=min_cluster_size, min_samples=min_samples,
        cluster_selection_epsilon=epsilon, metric=hdb_m,
        cluster_selection_method='eom')
    labels = clusterer.fit_predict(fit_input)
    logger.info(f'Clusters: {len(set(labels))-(1 if -1 in labels else 0)}  '
                f'Noise: {(labels==-1).sum()}')
    return labels, clusterer

def get_representative_samples(texts, labels, probabilities=None, n_samples=None):
    """
    Pick the most representative verbatims from each cluster.

    When HDBSCAN cluster probabilities are available, the top-N members of
    each cluster (by probability) are returned — these are the points
    HDBSCAN itself considers most prototypical of the cluster, so they are
    much better codebook-generation inputs than verbatims chosen by their
    arbitrary position in the source dataframe.

    Falls back to evenly-spaced selection only when no probabilities are
    supplied, which preserves backward compatibility with older callers.
    """
    if n_samples is None: n_samples = CLUSTER_REPRESENTATIVES
    reps = {}
    for cid in sorted(set(labels)):
        if cid == -1: continue
        if probabilities is not None:
            members = [(t, p) for t, l, p in zip(texts, labels, probabilities) if l == cid]
            members.sort(key=lambda tp: tp[1], reverse=True)
            take = min(n_samples, len(members))
            reps[cid] = [t for t, _ in members[:take]]
        else:
            ct  = [t for t, l in zip(texts, labels) if l == cid]
            idx = np.linspace(0, len(ct)-1, min(n_samples, len(ct)), dtype=int)
            reps[cid] = [ct[i] for i in idx]
    return reps

def discover_themes(df, text_column=None):
    """
    Orchestrate the full embedding → UMAP → HDBSCAN pipeline.
    All tuning parameters come from Section 0-B constants.

    Adds 'cluster' and 'cluster_probability' columns to df.

    Returns
    -------
    df      — with cluster columns added
    labels  — HDBSCAN integer labels (-1 = noise)
    reps    — {cluster_id: [verbatim, ...]} representative samples
    """
    if text_column is None: text_column = TEXT_COL

    # BUGFIX 3: preprocessing can leave a non-contiguous index after filtering.
    # Reset here so HDBSCAN labels/probabilities, dataframe rows, and later
    # .loc/.at operations all share a clean 0..n-1 row index.
    df = df.copy().reset_index(drop=True)

    texts = df[text_column].tolist()
    if len(texts) < 10: raise ValueError('Need ≥ 10 responses for clustering.')
    embeddings = generate_embeddings(texts)
    reduced    = reduce_dimensions_umap(embeddings)
    min_cs     = max(3, len(texts) // 50)
    labels, cl = cluster_responses(reduced, min_cluster_size=min_cs)
    probabilities = cl.probabilities_ if hasattr(cl,'probabilities_') else None
    reps = get_representative_samples(texts, labels, probabilities=probabilities)
    df['cluster']             = labels
    df['cluster_probability'] = probabilities if probabilities is not None else 0.0
    return df, labels, reps


### 2-B · Codebook generation and response coding

In [111]:
def _fmt_clusters(reps):
    """Format cluster verbatims as a text block for the LLM."""
    lines = []
    for cid, samples in reps.items():
        lines.append(f'\n### Cluster {cid+1}')
        for i, s in enumerate(samples, 1): lines.append(f'{i}. "{s}"')
    return '\n'.join(lines)

def generate_codebook(client, representatives, survey_context='',
                      target_codes=None, multi_label=True,
                      model=None, max_tokens=8000):
    """
    Generate a codebook from cluster representatives.

    This version is bilingual and more JSON-safe:
    - Uses CODEBOOK_LANGUAGE_CODE for French/English output.
    - Requests concise JSON so the response does not become too large.
    - Retries automatically with a larger token budget if the first JSON is
      truncated or invalid.
    """
    if model is None:
        model = REASONING_MODEL

    logger.info('Generating codebook (1 API call, with JSON retry if needed)...')
    ctx = survey_context or 'Open-ended survey responses.'
    lang_name = _codebook_language_name()
    other_name = get_default_other_code_name()
    ci = (f'Create exactly {target_codes} distinct codes' if target_codes
          else 'Create an appropriate number of distinct codes')
    li = ('The codes must support multi-label coding.' if multi_label
          else 'The codes must be mutually exclusive.')

    if target_codes and target_codes > 20 and globals().get('CODEBOOK_TARGET_CODES_CAP') is None:
        logger.warning(
            'Large codebook requested (%s codes). If JSON is still too long, set '
            'CODEBOOK_TARGET_CODES_CAP to a smaller number such as 12 or 15.',
            target_codes
        )

    base_umsg = (
        'Analyze the groups and create a qualitative codebook.\n\n'
        f'CODEBOOK OUTPUT LANGUAGE: {lang_name}.\n'
        f'Write code names, definitions, criteria, example verbatims, and the catch-all code in {lang_name}.\n\n'
        f'CONTEXT:\n{ctx}\n\nGROUPS:\n{_fmt_clusters(representatives)}\n\n'
        f'INSTRUCTIONS:\n1. {ci}.\n2. {li}\n'
        '3. Merge groups whose themes overlap.\n'
        f'4. Add this exact catch-all code: "{other_name}".\n'
        '5. Keep the JSON concise to avoid truncation:\n'
        '   - code name: max 5 words\n'
        '   - definition: max 25 words\n'
        '   - inclusion_criteria: 2 to 3 short items\n'
        '   - exclusion_criteria: 1 to 2 short items\n'
        '   - example_verbatims: max 2 short examples\n'
        '   - no extra fields\n\n'
        'Return ONLY valid JSON, with no Markdown and no text outside JSON:\n'
        '{"codes":[{"name":"...","definition":"...",'
        '"inclusion_criteria":["..."],"exclusion_criteria":["..."],'
        '"example_verbatims":["..."]}]}'
    )

    first_budget = max_tokens or globals().get('CODEBOOK_GENERATION_MAX_TOKENS', 8000)
    retry_budget = globals().get(
        'CODEBOOK_GENERATION_RETRY_MAX_TOKENS',
        max(first_budget, 12000)
    )
    token_budgets = [first_budget]
    if retry_budget and retry_budget > first_budget:
        token_budgets.append(retry_budget)

    last_error = None
    last_raw = ''

    for attempt, token_budget in enumerate(token_budgets, 1):
        retry_note = ''
        if attempt > 1:
            retry_note = (
                '\n\nIMPORTANT RETRY INSTRUCTION:\n'
                'The previous response was truncated or invalid JSON. Return a shorter, '
                'complete JSON object. Keep all text very concise and do not include markdown.'
            )

        resp = chat_with_retry(
            client, model=model,
            messages=[
                {'role': 'system', 'content': 'Qualitative research expert. Return ONLY complete valid JSON.'},
                {'role': 'user', 'content': base_umsg + retry_note}
            ],
            temperature=0.3,
            max_tokens=token_budget
        )

        choice = resp.choices[0]
        finish_reason = getattr(choice, 'finish_reason', '')
        last_raw = (choice.message.content or '').strip()

        try:
            if finish_reason == 'length':
                raise json.JSONDecodeError(
                    f'LLM output stopped because max_tokens={token_budget} was reached',
                    last_raw,
                    len(last_raw)
                )
            cb = json.loads(_extract_json(last_raw))
            _validate_codebook(cb)
            cb = ensure_other_code(cb, other_name=other_name)
            cb.update({
                'version': '1.0-draft',
                'description': survey_context,
                'multi_label': multi_label,
                'codebook_language': _codebook_language_code(),
                'generated_at': datetime.now(timezone.utc).isoformat(),
                'model': model
            })
            logger.info(f'Codebook generated: {len(cb["codes"])} codes.')
            return cb

        except (json.JSONDecodeError, ValueError, TypeError) as e:
            last_error = e
            logger.warning(
                'Codebook JSON parse/validation failed on attempt %s/%s with max_tokens=%s: %s',
                attempt, len(token_budgets), token_budget, e
            )

    tail = (last_raw or '')[-800:]
    raise ValueError(
        'Codebook generation returned truncated or invalid JSON after automatic retry. '
        'Recommended fixes: increase CODEBOOK_GENERATION_RETRY_MAX_TOKENS, reduce '
        'CODEBOOK_TARGET_CODES_CAP (for example 12 or 15), or reduce '
        'CLUSTER_REPRESENTATIVES to 1 or 2.\n\n'
        f'Last error: {last_error}\n\n'
        f'Last output tail for debugging:\n{tail}'
    )

def _deduplicate_texts(texts):
    """
    Return (unique_texts, index_map) to avoid coding identical responses twice.

    index_map[i] gives the position in unique_texts for original index i.
    After coding unique_texts, expand results with: [results[j] for j in index_map]

    Cost impact: identical responses are coded once. Result is reused for duplicates.
    For surveys with common short answers this can save 10-30% of coding calls.
    """
    seen, unique, idx_map = {}, [], []
    for t in texts:
        if t not in seen:
            seen[t] = len(unique)
            unique.append(t)
        idx_map.append(seen[t])
    saved = len(texts) - len(unique)
    if saved:
        logger.info(f'Pre-coding dedup: {len(texts)} -> {len(unique)} unique '
                    f'({saved} duplicate texts skip the API)')
    return unique, idx_map

def _coding_codebook_text(codebook):
    """Return the full or compact codebook text according to the cost setting."""
    if USE_COMPACT_CODEBOOK_FOR_CODING:
        return format_codebook_for_prompt_compact(codebook)
    return format_codebook_for_prompt(codebook)


OTHER_CODE_NAME = get_default_other_code_name()
OTHER_FALLBACK_CONFIDENCE = 0.50


def get_other_code_name(codebook, preferred_name=None):
    """
    Return the exact catch-all code name if present in the codebook.

    Matching is case-insensitive so user-supplied codebooks can already contain
    the category without creating a duplicate.
    """
    if preferred_name is None:
        preferred_name = get_default_other_code_name()
    preferred_norm = str(preferred_name).strip().lower()
    for code in codebook.get('codes', []):
        name = str(code.get('name', '')).strip()
        if name.lower() == preferred_norm:
            return name
    return None


def ensure_other_code(codebook, other_name=None, other_code_id=''):
    """
    Ensure a catch-all category exists in the codebook.

    If the codebook has user-supplied code IDs, the catch-all code must already
    exist in the source file or be given explicitly via `other_code_id`. The
    notebook does not invent IDs for governed/manual codebooks.
    """
    if 'codes' not in codebook or not isinstance(codebook['codes'], list):
        raise ValueError('Codebook must contain a list under key "codes".')

    if other_name is None:
        other_name = get_default_other_code_name()

    existing_other = get_other_code_name(codebook, other_name)
    if existing_other is not None:
        _validate_codebook(codebook)
        return codebook

    if codebook_requires_manual_ids(codebook) and not _clean_code_id(other_code_id):
        raise ValueError(
            f'Catch-all code "{other_name}" is missing. Because this codebook has '
            'a code ID column, add the catch-all row to the source codebook with a '
            'valid ID, or set S3_OTHER_CODE_ID before running Section 3.'
        )

    other_meta = _default_other_code_metadata(other_name)
    codebook['codes'].append(make_code_entry(
        other_name,
        code_id=other_code_id,
        definition=other_meta['definition'],
        inclusion_criteria=other_meta['inclusion_criteria'],
        exclusion_criteria=other_meta['exclusion_criteria'],
        example_verbatims=[],
        source='system_catch_all'
    ))
    _validate_codebook(codebook)
    logger.info(f'Added catch-all code: {other_name}')
    return codebook
def _coding_instruction(codebook):
    """Build coding instructions that prevent valid responses from staying uncoded."""
    ml = codebook.get('multi_label', True)
    other_code = get_other_code_name(codebook)

    if ml:
        base = 'Assign ALL applicable codes.'
    else:
        base = 'Assign EXACTLY ONE code.'

    if other_code:
        return (
            f'{base} If no specific code applies, assign "{other_code}". '
            'Use ONLY code names exactly as written in the codebook.'
        )

    return f'{base} Use ONLY code names exactly as written in the codebook.'


def _apply_other_fallback_if_needed(res, *, other_code, invalid_codes=None):
    """
    Add the catch-all category to a raw coding result when no valid code matched.

    The original model result is preserved through optional metadata so uncoded
    fallbacks remain auditable in `coding_result`.
    """
    if not other_code:
        return res

    if not isinstance(res, dict):
        res = {'codes': []}

    res = dict(res)
    codes = res.get('codes', [])
    if not isinstance(codes, list):
        codes = []

    res['codes'] = [{
        'code': other_code,
        'confidence': OTHER_FALLBACK_CONFIDENCE,
        'fallback': True
    }]
    res['fallback_reason'] = 'No valid code matched; assigned catch-all category.'
    if invalid_codes:
        res['invalid_returned_codes'] = invalid_codes
    return res

def _code_single(client, text, codebook, model=None):
    """
    Code one response. Used only as a fallback when a batch response cannot be parsed.

    Cost note: this is intentionally compact and omits reasoning to keep fallback calls cheap.
    """
    if model is None: model = CODING_MODEL
    ml = codebook.get('multi_label', True)
    schema = ('{"codes":[{"code":"<name>","confidence":0.0}],"reasoning":"brief"}'
              if SAVE_CODING_REASONING else
              '{"codes":[{"code":"<name>","confidence":0.0}]}')
    p  = (
        'Assign codes to this survey response.\n\n'
        f'CODEBOOK (use ONLY these codes):\n'
        f'{_coding_codebook_text(codebook)}\n\n'
        f'RESPONSE: {text}\n\n'
        f'INSTRUCTIONS: {_coding_instruction(codebook)}\n'
        'Include confidence 0.0-1.0. Return ONLY raw JSON:\n'
        f'{schema}'
    )
    r = chat_with_retry(client, model=model,
                        messages=[{'role':'system',
                                   'content':'Survey coder. Return ONLY raw JSON.'},
                                  {'role':'user', 'content':p}],
                        temperature=0.2, max_tokens=(450 if SAVE_CODING_REASONING else 250))
    c = re.sub(r'^```(?:json)?\n?', '', r.choices[0].message.content.strip())
    c = re.sub(r'\n?```$', '', c)
    try:    return json.loads(c)
    except: return {'codes':[]}

def _code_batch(client, texts, codebook, batch_size=None, model=None):
    """
    Code a list of texts in batches with cost optimisations:

    1. Pre-coding deduplication
       Identical texts are coded once; result is reused for all duplicates.

    2. Larger configurable batch size
       Defaults to CODING_BATCH_SIZE from 0-B (default 25 in v11).
       Higher = fewer repeated codebook prompts. Lower it if JSON parse errors occur.

    3. Compact codebook prompt
       USE_COMPACT_CODEBOOK_FOR_CODING=True sends code names + definitions only.

    4. Short output schema
       SAVE_CODING_REASONING=False removes the per-response reasoning field.

    5. Tight output token budget
       max_tokens scales with batch size and no longer assumes long reasoning text.
    """
    if model      is None: model      = CODING_MODEL
    if batch_size is None: batch_size = CODING_BATCH_SIZE
    unique_texts, idx_map = _deduplicate_texts(texts)
    unique_results = []
    ml = codebook.get('multi_label', True)
    instruction = _coding_instruction(codebook)
    schema = ('[{"response_num":1,"codes":[{"code":"<name>","confidence":0.0}],"reasoning":"brief"}]'
              if SAVE_CODING_REASONING else
              '[{"response_num":1,"codes":[{"code":"<name>","confidence":0.0}]}]')

    for i in range(0, len(unique_texts), batch_size):
        batch      = unique_texts[i:i+batch_size]
        bt         = ''.join(f'\n[{j}] "{t}"\n' for j, t in enumerate(batch, 1))
        per_response_budget = 160 if SAVE_CODING_REASONING else 90
        out_tokens = min(3500, max(400, len(batch) * per_response_budget))
        p = (
            'You are a precise qualitative survey coder.\n\n'
            f'CODEBOOK (use ONLY these codes):\n'
            f'{_coding_codebook_text(codebook)}\n\n'
            f'RESPONSES:\n{bt}\n\n'
            f'INSTRUCTIONS: {instruction}\n'
            'Include confidence 0.0-1.0.\n'
            'Return a JSON array -- one object per response, in the same order:\n'
            f'{schema}'
        )
        r = chat_with_retry(
            client, model=model,
            messages=[{'role':'system','content':'Return only valid JSON.'},
                      {'role':'user',  'content':p}],
            temperature=0.1, max_tokens=out_tokens)
        c = re.sub(r'^```(?:json)?\n?', '', r.choices[0].message.content.strip())
        c = re.sub(r'\n?```$', '', c)
        try:
            parsed = json.loads(c)
            if not isinstance(parsed, list) or len(parsed) != len(batch):
                raise ValueError(f'Expected {len(batch)} items, got {len(parsed) if isinstance(parsed, list) else type(parsed)}')
            unique_results.extend(parsed)
            time.sleep(0.3)
        except (json.JSONDecodeError, ValueError, TypeError) as e:
            logger.warning(f'Batch parse failed ({e}) -- single-coding {len(batch)} responses')
            for t in batch:
                unique_results.append(_code_single(client, t, codebook, model))
                time.sleep(0.2)
        logger.info(f'Coded {min(i+batch_size, len(unique_texts))}/{len(unique_texts)} unique')
    return [unique_results[j] for j in idx_map]

def apply_codes_to_dataframe(client, df, codebook, text_column=None, model=None):
    """
    Apply a codebook to every row of df using the configured CODING_MODEL.

    Adds code_<name> (0/1) and conf_<name> (0.0-1.0) for each code.
    Duplicate responses share a coding result (no repeated API calls).
    Run add_codes_column() after this to get the summary columns.
    """
    if text_column is None: text_column = TEXT_COL
    if model       is None: model       = CODING_MODEL   # high-volume classification
    n = len(df)
    logger.info(f'Coding {n} responses (batch={CODING_BATCH_SIZE}, model={model}, compact_codebook={USE_COMPACT_CODEBOOK_FOR_CODING})...')
    results = _code_batch(client, df[text_column].tolist(), codebook, model=model)
    for code in codebook['codes']:
        df[f"code_{code['name']}"] = 0
        df[f"conf_{code['name']}"] = 0.0
    other_code = get_other_code_name(codebook)
    other_col = f"code_{other_code}" if other_code else None
    other_conf_col = f"conf_{other_code}" if other_code else None
    normalized_results = []

    for idx, res in zip(df.index, results):
        assigned = 0
        invalid_codes = []
        if not isinstance(res, dict):
            res = {'codes': []}

        for ci in res.get('codes', []):
            code_name = str(ci.get('code', '')).strip() if isinstance(ci, dict) else ''
            cc_ = f"code_{code_name}"
            cf  = f"conf_{code_name}"
            if code_name and cc_ in df.columns:
                conf_value = _safe_confidence(ci.get('confidence', 0.0)) if isinstance(ci, dict) else 0.0
                df.at[idx, cc_] = 1
                df.at[idx, cf]  = conf_value
                if isinstance(ci, dict):
                    ci['confidence'] = conf_value
                assigned += 1
            elif code_name:
                invalid_codes.append(code_name)

        if assigned == 0 and other_code and other_col in df.columns:
            df.at[idx, other_col] = 1
            df.at[idx, other_conf_col] = OTHER_FALLBACK_CONFIDENCE
            res = _apply_other_fallback_if_needed(
                res, other_code=other_code, invalid_codes=invalid_codes
            )

        normalized_results.append(res)

    df['coding_result'] = [json.dumps(r, ensure_ascii=False) for r in normalized_results]
    code_cols = [f"code_{c['name']}" for c in codebook['codes']]
    uncoded_after_fallback = int((df[code_cols].sum(axis=1) == 0).sum())
    if uncoded_after_fallback:
        logger.warning(f'Coding complete with {uncoded_after_fallback} uncoded response(s).')
    else:
        logger.info('Coding complete. No uncoded responses remain after fallback handling.')
    return df


### 2-C · Run Section 2: No user-supplied codebook
*Use when you don't have a codebook and want AI to generate a codebook automatically*  
*Prerequisite: Section 1 complete, `df_preprocessed` exists.*

### 2-C-1 · Cluster responses

In [112]:
# ── Step 1: Cluster ──────────────────────────────────────────────────────
df_s2, labels, representatives = discover_themes(df_preprocessed.copy())
n_cl = len(set(labels)) - (1 if -1 in labels else 0)
print(f'Clusters: {n_cl}   Noise points: {(labels==-1).sum()}')


Embedding with text-embedding-3-small:   0%|          | 0/1 [00:00<?, ?it/s]

01:11:10 - INFO - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
Embedding with text-embedding-3-small: 100%|██████████| 1/1 [00:00<00:00,  1.31it/s]
01:11:10 - INFO - Embeddings: (40, 1536) | model=text-embedding-3-small | input_texts=40 | tokens=716
01:11:10 - INFO - UMAP: 1536D → 10D
01:11:10 - INFO - Clusters: 6  Noise: 2


Clusters: 6   Noise points: 2


### 2-C-2 · Generate AI codebook

In [113]:
# ── Step 2: Generate codebook ────────────────────────────────────────────
# Cost guardrails:
# - By default, target_codes follows the clustering result: len(representatives).
# - CODEBOOK_TARGET_CODES_CAP is optional. Use an integer cap only when you want
#   fewer, broader themes for cost control or executive reporting.
# - max_tokens uses CODEBOOK_GENERATION_MAX_TOKENS instead of a large hardcoded value.
target_codes_s2 = len(representatives)
cap_applied_s2 = CODEBOOK_TARGET_CODES_CAP is not None
if cap_applied_s2:
    target_codes_s2 = min(target_codes_s2, CODEBOOK_TARGET_CODES_CAP)

codebook_s2 = generate_codebook(
    client=client,
    representatives=representatives,
    survey_context=SURVEY_CONTEXT,
    target_codes=target_codes_s2,
    multi_label=True,
    max_tokens=CODEBOOK_GENERATION_MAX_TOKENS
)
display_codebook(codebook_s2)
save_codebook(codebook_s2, FILE_STEM + '_codebook_s2.json')

print(f"\nℹ️ Clusters/representative groups discovered: {len(representatives)}")
print(f"ℹ️ Codebook target codes used: {target_codes_s2}")
print(f"ℹ️ Codebook target cap: {CODEBOOK_TARGET_CODES_CAP if cap_applied_s2 else 'None — clustering-derived granularity preserved'}")
print(f"ℹ️ Codebook max output tokens: {CODEBOOK_GENERATION_MAX_TOKENS}")


01:11:10 - INFO - Generating codebook (1 API call, with JSON retry if needed)...
01:11:14 - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
01:11:14 - INFO - LLM usage | model=gpt-5.4-mini | prompt=664 | completion=628 | total=1292 | cached=0
01:11:14 - INFO - Codebook generated: 6 codes.
01:11:14 - INFO - Codebook saved → PREF1_DEGUSTATION_AM_FESTI237_codebook_s2.json



CODEBOOK  version=1.0-draft  codes=6  multi_label=True  has_code_ids=False

[1] Goût sucré et agréable
    Definition : Préférence liée à un goût sucré, doux ou globalement agréable en bouche.
    Include    : Goût sucré; Goût doux ou agréable; Saveur appréciée
    Exclude    : Fraîcheur sans mention du goût; Teneur en alcool
    Example    : Le goût est tellement sucré...

[2] Fraîcheur et rafraîchissement
    Definition : Préférence fondée sur l'effet rafraîchissant, la sensation de fraîcheur ou de boisson désaltérante.
    Include    : Rafraîchissant; Parfum frais; Sensation de fraîcheur
    Exclude    : Goût sucré seul; Prix ou marque
    Example    : Ça a bon goût et c'est très rafraîchissant...

[3] Arômes et saveurs fruitées
    Definition : Préférence pour une saveur, un arôme ou un mélange aromatique spécifique, souvent fruité ou parfumé.
    Include    : Saveur ananas, grenadine, orange; Mélange d'arômes; Parfum ou arôme apprécié
    Exclude    : Sucré sans arôme précis; Fra

### 2-C-3 · Code responses, generate reports, and save reset baseline

In [114]:
# ── Step 3: Code responses + generate reports ────────────────────────────
df_s2 = apply_codes_to_dataframe(client, df_s2, codebook_s2)
df_s2 = add_codes_column(df_s2, codebook_s2)
outputs_s2 = generate_reports(df_s2, codebook_s2,
    output_dir='reports_s2/', output_file=FILE_STEM+'_s2.xlsx')

# Create the clean Section 2 reset baseline BEFORE any split/merge refinements.
set_pipeline_baseline(
    's2', df_s2, codebook_s2,
    metadata={
        'section': 'Section 2',
        'description': 'Clean Section 2 output: AI-generated codebook + coded responses before split/merge.',
        'outputs': outputs_s2,
    }
)

print('\n✅ Section 2 complete!')
print(f'   Codes : {[c["name"] for c in codebook_s2["codes"]]}')
print(f'   Files : {outputs_s2}')
print('   Reset : baseline saved as s2; use reset_to_section2() to restore it.')

# ── To continue with Section 4 or 5: ─────────────────────────────────────
df_current, codebook_current = df_s2.copy(deep=True), copy.deepcopy(codebook_s2)


01:11:14 - INFO - Coding 40 responses (batch=10, model=gpt-4.1-mini, compact_codebook=True)...
01:11:21 - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
01:11:21 - INFO - LLM usage | model=gpt-4.1-mini | prompt=517 | completion=602 | total=1119 | cached=0
01:11:21 - INFO - Coded 10/40 unique
01:11:29 - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
01:11:29 - INFO - LLM usage | model=gpt-4.1-mini | prompt=472 | completion=514 | total=986 | cached=0
01:11:30 - INFO - Coded 20/40 unique
01:11:36 - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
01:11:36 - INFO - LLM usage | model=gpt-4.1-mini | prompt=499 | completion=593 | total=1092 | cached=0
01:11:37 - INFO - Coded 30/40 unique
01:11:44 - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
01:11:44 - INFO - LLM usage | model=gpt-4.1-mini | prompt=464 | completion=574 | total=1038 | 

✅ Reset baseline set: s2 — 40 rows, 6 codes

✅ Section 2 complete!
   Codes : ['Goût sucré et agréable', 'Fraîcheur et rafraîchissement', 'Arômes et saveurs fruitées', 'Équilibre du produit', 'Teneur en alcool', 'Autre / Non classifiable']
   Files : {'frequency_chart': 'reports_s2\\PREF1_DEGUSTATION_AM_FESTI237_frequencies.png', 'cooccurrence_chart': 'reports_s2\\PREF1_DEGUSTATION_AM_FESTI237_cooccurrence.png', 'summary_report': 'reports_s2\\PREF1_DEGUSTATION_AM_FESTI237_s2.xlsx'}
   Reset : baseline saved as s2; use reset_to_section2() to restore it.


---
## SECTION 3 · User-Supplied Codebook
*Use when you already have a codebook — even one with only code names, no definitions.*  
*Prerequisite: Section 1 complete, `df_preprocessed` exists.*

### Processing pipeline
```
Your codebook file (.xlsx / .csv / .json / .txt)  OR  a Python list
  ↓  load_user_codebook()       — read code names and optional code IDs + definitions, build minimal dict
  ↓  enrich_codebook_with_ai()  — (optional) configured LLM writes definitions & criteria
  ↓  apply_codes_to_dataframe() — configured LLM codes every response
  ↓  generate_reports()         — Excel + charts
  ↓
(df_s3, codebook_s3)  →  assign to df_current, codebook_current for Sections 4/5
```

### Accepted input formats
| Format | Notes |
|--------|-------|
| Excel `.xlsx` | Set `S3_CODEBOOK_SHEET`, `S3_CODEBOOK_COL`, optional `S3_CODEBOOK_ID_COL`, and optional `S3_CODEBOOK_DEFINITION_COL` |
| CSV `.csv` | Set `S3_CODEBOOK_COL`, optional `S3_CODEBOOK_ID_COL`, and optional `S3_CODEBOOK_DEFINITION_COL` |
| JSON `.json` | List of strings OR `[{"name": "..."}]` |
| Plain text `.txt` | One code per line |
| Python list | Assign directly to `S3_CODEBOOK_SOURCE` |


> **Cost reminder**  
> Run Section 2 **or** Section 3 for a normal production run. Running both full sections will code the dataset twice and roughly doubles the response-coding cost.

> **Code ID rule**  
> If your imported codebook has a code ID column, the notebook preserves it. Any new catch-all, split, or merged code must receive a manually supplied ID; the notebook will not invent one.

> If your imported codebook has a definition column, the notebook preserves it. `S3_ENRICH_WITH_AI=True` only fills missing definitions; it does not replace definitions already supplied in the codebook.


### 3-A · Section 3 configuration

In [115]:
S3_CODEBOOK_FILE   = 'sample_codebook_test.xlsx'
S3_CODEBOOK_SHEET  = 'Codebook'    # sheet name (Excel only)
S3_CODEBOOK_COL    = 'Code Name'   # column containing code names
# S3_CODEBOOK_ID_COL = None          # optional column containing stable code IDs, e.g. 'Code ID'
S3_CODEBOOK_ID_COL = "Code ID"          # optional column containing stable code IDs, e.g. 'Code ID'
# S3_CODEBOOK_DEFINITION_COL = None  # optional column containing code definitions, e.g. 'Definition'
S3_CODEBOOK_DEFINITION_COL = 'Definition'  # optional column containing code definitions, e.g. 'Definition'

# To use a Python list instead, assign it here:
# S3_CODEBOOK_SOURCE = ['Goût agréable', 'Prix abordable', 'Autre']
S3_CODEBOOK_SOURCE = S3_CODEBOOK_FILE  # ← change to a list to override

# Required only when S3_CODEBOOK_ID_COL is set and the catch-all code is not already
# present in the source codebook. The notebook will not invent governance IDs.
S3_OTHER_CODE_ID = 'OTHER'

# Enrich with AI: recommended when codes have no definitions
S3_ENRICH_WITH_AI = True

S3_OUTPUT_DIR  = 'reports_s3/'
S3_OUTPUT_FILE = FILE_STEM + '_s3.xlsx'


### 3-B · Load user codebook function

In [116]:
def load_user_codebook(source, name_column=None, id_column=None,
                       definition_column=None, sheet_name=0):
    """
    Load a user-supplied codebook with required code names plus optional IDs and definitions.

    Parameters
    ----------
    source            : str | list  — file path, list of code names, or list of dict entries
    name_column       : str | None  — column with code names (Excel/CSV)
    id_column         : str | None  — optional column with stable/manual code IDs
    definition_column : str | None  — optional column with user-provided definitions
    sheet_name        : str | int   — sheet to read (Excel only)

    Returns
    -------
    dict — minimal codebook compatible with the full pipeline
    """
    entries = []

    def _entry_from_row(row, name_col, id_col=None, definition_col=None):
        name = _clean_code_name(row[name_col])
        if not name:
            return None
        code_id = _clean_code_id(row[id_col]) if id_col else ''
        if id_col and not code_id:
            raise ValueError(f'Code "{name}" is missing an ID in column "{id_col}".')
        definition = _clean_optional_text(row[definition_col]) if definition_col else ''
        return make_code_entry(
            name,
            code_id=code_id,
            definition=definition,
            source='user_codebook'
        )

    def _definition_from_dict(item):
        return _clean_optional_text(
            item.get('definition') or
            item.get('Definition') or
            item.get('Définition') or
            item.get('Code Definition') or
            item.get('code_definition') or
            ''
        )

    if isinstance(source, list):
        for item in source:
            if isinstance(item, dict):
                name = _clean_code_name(item.get('name') or item.get('Code Name') or item.get('code'))
                code_id = _clean_code_id(item.get(CODE_ID_FIELD) or item.get('Code ID') or item.get('id'))
                if name:
                    entries.append(make_code_entry(
                        name,
                        code_id=code_id,
                        definition=_definition_from_dict(item),
                        inclusion_criteria=item.get('inclusion_criteria',[]),
                        exclusion_criteria=item.get('exclusion_criteria',[]),
                        example_verbatims=item.get('example_verbatims',[]),
                        source='user_codebook'
                    ))
            else:
                name = _clean_code_name(item)
                if name:
                    entries.append(make_code_entry(name, source='user_codebook'))
    elif isinstance(source, str):
        path = Path(source)
        if path.suffix.lower() in ('.xlsx','.xls'):
            df_ = pd.read_excel(path, sheet_name=sheet_name)
            col = name_column or df_.columns[0]
            if col not in df_.columns:
                raise ValueError(f'name_column "{col}" not found. Available columns: {list(df_.columns)}')
            if id_column and id_column not in df_.columns:
                raise ValueError(f'id_column "{id_column}" not found. Available columns: {list(df_.columns)}')
            if definition_column and definition_column not in df_.columns:
                raise ValueError(f'definition_column "{definition_column}" not found. Available columns: {list(df_.columns)}')
            for _, row in df_.dropna(subset=[col]).iterrows():
                entry = _entry_from_row(row, col, id_column, definition_column)
                if entry: entries.append(entry)
        elif path.suffix.lower() == '.csv':
            df_ = pd.read_csv(path)
            col = name_column or df_.columns[0]
            if col not in df_.columns:
                raise ValueError(f'name_column "{col}" not found. Available columns: {list(df_.columns)}')
            if id_column and id_column not in df_.columns:
                raise ValueError(f'id_column "{id_column}" not found. Available columns: {list(df_.columns)}')
            if definition_column and definition_column not in df_.columns:
                raise ValueError(f'definition_column "{definition_column}" not found. Available columns: {list(df_.columns)}')
            for _, row in df_.dropna(subset=[col]).iterrows():
                entry = _entry_from_row(row, col, id_column, definition_column)
                if entry: entries.append(entry)
        elif path.suffix.lower() == '.json':
            with open(path, encoding='utf-8') as f:
                data = json.load(f)
            for item in data:
                if isinstance(item, dict):
                    name = _clean_code_name(item.get('name') or item.get('Code Name') or item.get('code'))
                    code_id = _clean_code_id(item.get(CODE_ID_FIELD) or item.get('Code ID') or item.get('id'))
                    if name:
                        entries.append(make_code_entry(
                            name,
                            code_id=code_id,
                            definition=_definition_from_dict(item),
                            inclusion_criteria=item.get('inclusion_criteria',[]),
                            exclusion_criteria=item.get('exclusion_criteria',[]),
                            example_verbatims=item.get('example_verbatims',[]),
                            source='user_codebook'
                        ))
                else:
                    name = _clean_code_name(item)
                    if name: entries.append(make_code_entry(name, source='user_codebook'))
        elif path.suffix.lower() == '.txt':
            if id_column:
                raise ValueError('Plain text codebooks cannot preserve code IDs. Use Excel, CSV, or JSON.')
            if definition_column:
                raise ValueError('Plain text codebooks cannot preserve definitions. Use Excel, CSV, or JSON.')
            with open(path, encoding='utf-8') as f:
                entries = [make_code_entry(l.strip(), source='user_codebook')
                           for l in f if l.strip()]
        else:
            raise ValueError(f'Unsupported format: {path.suffix}')
    else:
        raise TypeError('source must be a file path or a list of code names/dicts.')

    if not entries:
        raise ValueError('No codes found. Check your file and name_column.')

    # Dedupe and validate. Names must be unique because coded columns are name-based.
    seen_names, unique = set(), []
    for entry in entries:
        name_key = entry['name'].strip().lower()
        if name_key in seen_names:
            raise ValueError(f'Duplicate code name found: "{entry["name"]}". Code names must be unique.')
        seen_names.add(name_key)
        unique.append(entry)

    codebook = {'version':'user-supplied','multi_label':True,
                'description':'','generated_at':None,'model':None,
                'code_id_column': id_column,
                'definition_column': definition_column,
                'codes': unique}
    _validate_codebook(codebook)
    definitions_preserved = sum(bool(_clean_optional_text(c.get('definition'))) for c in unique)
    logger.info(
        f'Loaded {len(unique)} codes. code_ids_preserved={codebook_has_code_ids(codebook)} '
        f'definitions_preserved={definitions_preserved}'
    )
    return codebook


### 3-C · Run Section 3

### 3-C-1 · Load and optionally enrich the user codebook

In [117]:
# ── Step 1: Load codebook ────────────────────────────────────────────────
codebook_s3 = load_user_codebook(
    source=S3_CODEBOOK_SOURCE, name_column=S3_CODEBOOK_COL,
    id_column=S3_CODEBOOK_ID_COL,
    definition_column=S3_CODEBOOK_DEFINITION_COL,
    sheet_name=S3_CODEBOOK_SHEET)
codebook_s3 = ensure_other_code(codebook_s3, other_code_id=S3_OTHER_CODE_ID)

# ── Step 2: Enrich definitions (skip if S3_ENRICH_WITH_AI = False) ────────
if S3_ENRICH_WITH_AI:
    codebook_s3 = enrich_codebook_with_ai(client, codebook_s3, SURVEY_CONTEXT)

display_codebook(codebook_s3)
save_codebook(codebook_s3, FILE_STEM + '_codebook_s3.json')


01:11:45 - INFO - Loaded 10 codes. code_ids_preserved=True definitions_preserved=8
01:11:45 - INFO - Enriching 2 codes (1 API call)...
01:11:49 - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
01:11:49 - INFO - LLM usage | model=gpt-4.1-mini | prompt=298 | completion=297 | total=595 | cached=0
01:11:49 - INFO - Enrichment complete. Matched definitions for: ['Prix abordable', 'Habitude / fidélité']
01:11:49 - INFO - Codebook saved → PREF1_DEGUSTATION_AM_FESTI237_codebook_s3.json



CODEBOOK  version=user-supplied  codes=10  multi_label=True  has_code_ids=True

[1] [C001] Goût agréable
    Definition : Definition1

[2] [C002] Rafraîchissement
    Definition : Definition2

[3] [C003] Prix abordable
    Definition : Ce code regroupe les réponses où le consommateur met en avant le coût raisonnable ou accessible de la boisson comme raison principale de sa préférence.
    Include    : Mention explicite du prix bas ou abordable; Référence à un bon rapport qualité-prix; Comparaison favorable du prix avec d'autres boissons
    Exclude    : Références uniquement à la qualité sans mention du prix; Commentaires sur la marque sans lien avec le prix
    Example    : J'aime cette boisson parce qu'elle est moins chère que les autres....

[4] [C004] Disponibilité
    Definition : Definition4

[5] [C005] Marque connue / fiable
    Definition : Definition5

[6] [C006] Habitude / fidélité
    Definition : Ce code concerne les réponses où la préférence pour la boisson est liée à une

### 3-C-2 · Code responses, generate reports, and save reset baseline

In [118]:
# ── Step 3: Code responses + generate reports ────────────────────────────
df_s3 = apply_codes_to_dataframe(client, df_preprocessed.copy(), codebook_s3)
df_s3 = add_codes_column(df_s3, codebook_s3)
outputs_s3 = generate_reports(df_s3, codebook_s3,
    output_dir=S3_OUTPUT_DIR, output_file=S3_OUTPUT_FILE)

# Create the clean Section 3 reset baseline BEFORE any split/merge refinements.
set_pipeline_baseline(
    's3', df_s3, codebook_s3,
    metadata={
        'section': 'Section 3',
        'description': 'Clean Section 3 output: user-supplied/enriched codebook + coded responses before split/merge.',
        'outputs': outputs_s3,
    }
)

print('\n✅ Section 3 complete!')
print(f'   Codes : {[c["name"] for c in codebook_s3["codes"]]}')
print(f'   Files : {outputs_s3}')
print('   Reset : baseline saved as s3; use reset_to_section3() to restore it.')

# ── To continue with Section 4 or 5: ─────────────────────────────────────
df_current, codebook_current = df_s3.copy(deep=True), copy.deepcopy(codebook_s3)


01:11:49 - INFO - Coding 40 responses (batch=10, model=gpt-4.1-mini, compact_codebook=True)...
01:11:56 - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
01:11:56 - INFO - LLM usage | model=gpt-4.1-mini | prompt=531 | completion=486 | total=1017 | cached=0
01:11:56 - INFO - Coded 10/40 unique
01:12:01 - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
01:12:01 - INFO - LLM usage | model=gpt-4.1-mini | prompt=486 | completion=244 | total=730 | cached=0
01:12:01 - INFO - Coded 20/40 unique
01:12:05 - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
01:12:05 - INFO - LLM usage | model=gpt-4.1-mini | prompt=513 | completion=232 | total=745 | cached=0
01:12:05 - INFO - Coded 30/40 unique
01:12:10 - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
01:12:10 - INFO - LLM usage | model=gpt-4.1-mini | prompt=478 | completion=438 | total=916 | ca

✅ Reset baseline set: s3 — 40 rows, 10 codes

✅ Section 3 complete!
   Codes : ['Goût agréable', 'Rafraîchissement', 'Prix abordable', 'Disponibilité', 'Marque connue / fiable', 'Habitude / fidélité', 'Effet énergisant', 'Socialisation', 'Santé / Naturalité', 'Autre / Non classifiable']
   Files : {'frequency_chart': 'reports_s3\\PREF1_DEGUSTATION_AM_FESTI237_frequencies.png', 'cooccurrence_chart': 'reports_s3\\PREF1_DEGUSTATION_AM_FESTI237_cooccurrence.png', 'summary_report': 'reports_s3\\PREF1_DEGUSTATION_AM_FESTI237_s3.xlsx'}
   Reset : baseline saved as s3; use reset_to_section3() to restore it.


### 3-D · Inspect results

### 3-D-1 · Frequency summary

In [119]:
freq_s3 = calculate_frequencies(df_s3, codebook_s3)
print('Frequencies (Section 3):'); print(freq_s3.to_string(index=False))


Frequencies (Section 3):
code_id                     code  count  percentage
   C001            Goût agréable     34        85.0
   C002         Rafraîchissement      4        10.0
   C010 Autre / Non classifiable      4        10.0
   C009       Santé / Naturalité      2         5.0
   C003           Prix abordable      0         0.0
   C004            Disponibilité      0         0.0
   C006      Habitude / fidélité      0         0.0
   C005   Marque connue / fiable      0         0.0
   C008            Socialisation      0         0.0
   C007         Effet énergisant      0         0.0


### 3-D-2 · Sample coded rows

In [120]:
cc_s3 = [c for c in df_s3.columns if c.startswith('code_')]
df_s3[['response_id','original_text','codes','num_codes'] + cc_s3].head(10)


,response_id,original_text,codes,num_codes,code_Goût agréable,code_Rafraîchissement,code_Prix abordable,code_Disponibilité,code_Marque connue / fiable,code_Habitude / fidélité,code_Effet énergisant,code_Socialisation,code_Santé / Naturalité,code_Autre / Non classifiable,code_ids
0,1,Parce que son goût est bien sucré et c'est raf...,Goût agréable | Rafraîchissement,2,1,1,0,0,0,0,0,0,0,0,C001 | C002
1,2,It taste good and it's very refresshing,Goût agréable | Rafraîchissement,2,1,1,0,0,0,0,0,0,0,0,C001 | C002
2,3,Parce que j'aime son goût de grenadine et c'es...,Goût agréable,1,1,0,0,0,0,0,0,0,0,0,C001
3,4,The taste is so sweet,Goût agréable,1,1,0,0,0,0,0,0,0,0,0,C001
4,5,"Parce que j'aime son goût de grenadine, son go...",Goût agréable,1,1,0,0,0,0,0,0,0,0,0,C001
5,6,C'est rafraîchissant et son goût est bon et c'...,Goût agréable | Rafraîchissement,2,1,1,0,0,0,0,0,0,0,0,C001 | C002
6,7,Le taux de sucre est moins élevé que Les autre...,Santé / Naturalité,1,0,0,0,0,0,0,0,0,1,0,C009
7,8,"Le goût est aromatisé, le parfum sent bon , el...",Goût agréable,1,1,0,0,0,0,0,0,0,0,0,C001
8,9,Je préfère cette boisson parce que son goût es...,Goût agréable,1,1,0,0,0,0,0,0,0,0,0,C001
10,10,À cause de son odeur de sucre et côté arrière ...,Goût agréable,1,1,0,0,0,0,0,0,0,0,0,C001


### 3-D-3 · Multi-label and uncoded QA checks

In [121]:
print(f'Multi-label responses : {df_s3["has_multi_label"].sum()}')
print(f'Avg codes per response: {df_s3["num_codes"].mean():.2f}')
uncoded_s3 = df_s3[df_s3["num_codes"] == 0].copy()
print(f'Uncoded responses     : {len(uncoded_s3)}')

if len(uncoded_s3):
    print('\n⚠️ Review these rows: they were not assigned any valid code.')
    display_cols = [c for c in ['response_id', 'original_text', 'coding_result'] if c in uncoded_s3.columns]
    display(uncoded_s3[display_cols].head(25))
else:
    print(f'✅ All valid Section 3 responses have at least one code. Responses that do not fit a specific code are assigned to {get_default_other_code_name()}.')


Multi-label responses : 4
Avg codes per response: 1.10
Uncoded responses     : 0
✅ All valid Section 3 responses have at least one code. Responses that do not fit a specific code are assigned to Autre / Non classifiable.


---
## SECTION 4 · Split a Code into Sub-Codes
*Use when a code is too broad and captures meaningfully different ideas.*  
*Prerequisite: a coded `(df_current, codebook_current)` pair from Section 2, 3, or a previous Section 4/5.*

### When to split
- Code frequency is high but verbatims feel heterogeneous
- A code name covers multiple distinct concepts (e.g. "Goût" = sweetness + bitterness + aroma)
- The code rarely co-occurs with others — it is self-contained but too broad

### What this section does
```
(df_current, codebook_current)  +  one parent code
  ↓
  Option A — provide sub-code names, IDs, and/or definitions manually
  Option B — leave them blank and let the notebook generate what is missing
       • S4_SUBCODES = []              → AI proposes sub-code names
       • S4_SUBCODE_IDS = []           → notebook auto-generates stable IDs when the codebook uses IDs
       • S4_SUBCODE_DEFINITIONS = []   → AI writes missing definitions when S4_ENRICH=True
  ↓  enrich_codebook_with_ai()   — write missing definitions for sub-codes
  ↓  _recode_affected_rows()     — configured LLM re-codes ONLY rows carrying the parent
  ↓  Remove parent columns; add sub-code columns in same codebook position
  ↓  generate_reports()
  ↓
df_current, codebook_current  updated in place  → chain into Section 5 or another Section 4
```

### Code ID behavior
- Existing user-provided code IDs are preserved.
- If the current codebook has IDs, every split-derived code also gets an ID.
- Manual `S4_SUBCODE_IDS` are used when provided.
- Missing `S4_SUBCODE_IDS` are generated deterministically from the parent code and sub-code names, then collision-checked.

### API cost note
This section can make 2–3 LLM calls: one to propose sub-code names if `S4_SUBCODES=[]`, one to enrich missing definitions if `S4_ENRICH=True`, and one or more calls to recode only affected rows.


### 4-A · Section 4 configuration

In [122]:
# df_current and codebook_current flow in automatically from Section 2-C or 3-C.
# For reset/restore commands and checkpoint rules, see Section 0-G.

# ── Section 4 configuration: split one OR many parent codes ───────────────
# Each item below is one split rule.
#
# Required:
#   parent_code: the existing code to split. By default, the parent code is REMOVED and replaced by its subcodes.
#
# Manual split option:
#   subcodes: provide 2+ sub-code names.
#
# AI split option:
#   subcodes: [] and set n_subcodes_proposed to the number of sub-codes needed.
#
# Optional:
#   subcode_ids: manual IDs. Use [] or blanks to auto-generate when IDs are used.
#   subcode_definitions: one definition per subcode, or [] with enrich=True.
#   n_subcodes_proposed: number of AI-proposed sub-codes when subcodes=[]
#   enrich: whether AI writes missing definitions
#
# Example:
# S4_SPLIT_CONFIGS = [
#     {
#         'parent_code': 'Parent code1',
#         'subcodes': ['Parent code1 - subcode A', 'Parent code1 - subcode B'],
#         'subcode_ids': [],
#         'subcode_definitions': [],
#         'n_subcodes_proposed': 2,
#         'enrich': True
#     },
#     {
#         'parent_code': 'Parent code2',
#         'subcodes': [],               # AI proposes names
#         'subcode_ids': [],
#         'subcode_definitions': [],
#         'n_subcodes_proposed': 3,      # Parent code2 split into 3
#         'enrich': True
#     }
# ]

S4_SPLIT_CONFIGS = [
    {
        'parent_code': 'Goût agréable',
        'subcodes': [],               # [] = AI proposes the sub-codes
        'subcode_ids': [],
        'subcode_definitions': [],
        'n_subcodes_proposed': 3,
        'enrich': True
    }
]

# Parent code preservation:
# False = remove each selected parent code and replace it with its sub-codes.
# True  = keep the original parent code column and keep the parent in the codebook,
#         then add sub-codes underneath it.
S4_PRESERVE_PARENT = False

S4_OUTPUT_DIR  = 'reports_s4/'
S4_OUTPUT_FILE = FILE_STEM + '_s4_split.xlsx'


### 4-B · Split function

In [123]:
def split_code(df, codebook, client, parent_code,
               subcodes=None, subcode_ids=None, subcode_definitions=None,
               n_subcodes_proposed=3, enrich_subcodes=True,
               survey_context='', text_column=None,
               output_dir='reports_s4/', output_file='split.xlsx',
               preserve_parent=False):
    """
    Split parent_code into two or more finer sub-codes.

    By default, the parent code is removed from the dataframe/codebook and replaced by subcodes.
    Set preserve_parent=True only when you want to keep the parent beside the subcodes.

    Manual inputs are respected when provided. Missing split-derived fields are
    generated automatically:
      - names: AI proposes them when subcodes is blank
      - IDs: deterministic notebook-generated IDs when the current codebook uses IDs
      - definitions: AI enrichment when enrich_subcodes=True

    Existing user-provided code IDs and definitions are preserved.
    """
    if text_column is None: text_column = TEXT_COL
    ctx = survey_context or SURVEY_CONTEXT
    subcodes = subcodes or []
    subcode_ids = subcode_ids or []
    subcode_definitions = subcode_definitions or []
    logger.info('='*60)
    logger.info(f'SECTION 4: Splitting "{parent_code}"')
    logger.info('='*60)
    pe = next((c for c in codebook['codes'] if c['name']==parent_code), None)
    if pe is None:
        raise ValueError(f'"{parent_code}" not in codebook. '
                         f'Available: {[c["name"] for c in codebook["codes"]]}')

    requires_ids = codebook_requires_manual_ids(codebook)
    split_source = 'manual_split' if subcodes else 'ai_split_proposal'

    # 1. Determine sub-code names. If none are provided, ask AI to propose them.
    if subcodes and len(subcodes) >= 2:
        final_sc = [_clean_code_name(s) for s in subcodes if _clean_code_name(s)]
        logger.info(f'User-defined sub-codes: {final_sc}')
    else:
        pc = f'code_{parent_code}'
        if pc not in df.columns:
            raise KeyError(f'"{pc}" not found. Run coding first.')
        affected = df.loc[df[pc]==1, text_column].tolist()
        if len(affected) < 2:
            raise ValueError(f'Only {len(affected)} response(s) carry "{parent_code}". '
                             'Need ≥ 2 to split.')
        sample = affected[:30]
        vb = '\n'.join(f'{i+1}. "{r}"' for i,r in enumerate(sample))
        lang_name = _codebook_language_name()
        p  = (
            'You are a qualitative research expert.\n\n'
            f'CODEBOOK OUTPUT LANGUAGE: {lang_name}.\n'
            f'Generate the sub-code names in {lang_name}.\n\n'
            f'CONTEXT: {ctx}\n\n'
            f'The code "{parent_code}" is too broad. '
            f'Here are {len(sample)} responses assigned to it:\n{vb}\n\n'
            f'Propose exactly {n_subcodes_proposed} distinct and mutually exclusive sub-codes '
            f'with short names in {lang_name}.\n'
            'Definitions will be generated later by AI enrichment.\n'
            'Return ONLY valid JSON:\n'
            '{"subcodes": ["Name 1", "Name 2", "..."]}'
        )
        # REASONING_MODEL: deciding how to split requires conceptual judgement
        r = chat_with_retry(
            client, model=REASONING_MODEL,
            messages=[{'role':'system','content':'Qualitative coding expert. Return ONLY valid JSON.'},
                      {'role':'user',  'content':p}],
            temperature=0.4, max_tokens=500)
        try:
            final_sc = [s.strip() for s in
                        json.loads(_extract_json(r.choices[0].message.content.strip()))
                        .get('subcodes',[]) if s.strip()]
        except Exception as e:
            logger.error(f'Sub-code proposal parse error: {e}'); final_sc = []
        if not final_sc:
            raise ValueError('Model returned no sub-codes. Set S4_SUBCODES manually.')
        logger.info(f'AI proposed sub-codes: {final_sc}')

    if len(final_sc) < 2:
        raise ValueError('Need ≥ 2 sub-codes to split.')
    if len(set(s.lower() for s in final_sc)) != len(final_sc):
        raise ValueError(f'Duplicate sub-code names are not allowed: {final_sc}')

    existing_names_lower = {
        c['name'].lower() for c in codebook['codes']
        if c.get('name', '').lower() != parent_code.lower()
    }
    name_collisions = [sc for sc in final_sc if sc.lower() in existing_names_lower]
    if name_collisions:
        raise ValueError(
            f'Sub-code name(s) already exist in the codebook: {name_collisions}. '
            'Use unique sub-code names, for example prefix them with the parent code.'
        )

    # 2. Determine sub-code IDs. Preserve manual IDs; auto-generate missing IDs
    # when the current codebook already uses IDs.
    manual_ids = _normalize_manual_id_list(subcode_ids, len(final_sc), 'S4_SUBCODE_IDS')
    if requires_ids:
        existing_ids = _existing_code_ids(codebook)
        parent_id = _clean_code_id(pe.get(CODE_ID_FIELD, ''))
        parent_prefix = parent_id or _slugify_for_code_id(parent_code, default='SPLIT', max_len=24)
        final_ids = []
        for sc, manual_id in zip(final_sc, manual_ids):
            if manual_id:
                if manual_id in existing_ids:
                    raise ValueError(f'S4_SUBCODE_IDS contains an ID already in the codebook: {manual_id}')
                existing_ids.add(manual_id)
                final_ids.append(manual_id)
            else:
                final_ids.append(_auto_code_id_for_new_code(
                    codebook, sc, prefix=parent_prefix, existing_ids=existing_ids
                ))
        logger.info(f'Sub-code IDs: {list(zip(final_ids, final_sc))}')
    else:
        if any(manual_ids):
            logger.warning('S4_SUBCODE_IDS were provided but the current codebook has no IDs; ignoring them.')
        final_ids = [''] * len(final_sc)
        parent_id = ''

    if final_ids and len(set(x for x in final_ids if x)) != len([x for x in final_ids if x]):
        raise ValueError(f'Duplicate sub-code IDs are not allowed: {final_ids}')

    # 3. Determine definitions. Manual definitions are preserved; missing ones
    # are AI-generated when enrich_subcodes=True.
    final_defs = [_clean_optional_text(x) for x in subcode_definitions]
    if final_defs and len(final_defs) != len(final_sc):
        raise ValueError('S4_SUBCODE_DEFINITIONS must be blank or have the same length as S4_SUBCODES.')
    if not final_defs:
        final_defs = [''] * len(final_sc)
    missing_defs = [sc for sc, definition in zip(final_sc, final_defs) if not definition]
    if missing_defs and not enrich_subcodes:
        raise ValueError(
            'Split-derived codes require definitions. Provide S4_SUBCODE_DEFINITIONS '
            'for every sub-code or set S4_ENRICH=True so AI generates the missing definitions. '
            f'Missing definitions for: {missing_defs}'
        )

    # 4. Build minimal sub-code codebook
    sc_cb = {'version':'split-draft','multi_label':codebook.get('multi_label',True),
             'codes':[make_code_entry(sc, code_id=cid, definition=definition,
                       inclusion_criteria=[], exclusion_criteria=[], example_verbatims=[],
                       parent_code=parent_code, parent_code_id=parent_id,
                       source=split_source)
                      for sc, cid, definition in zip(final_sc, final_ids, final_defs)]}
    _validate_codebook(sc_cb)

    # 5. Enrich missing sub-code definitions. If the LLM response is incomplete
    # or uses slightly different labels, fall back to conservative deterministic
    # definitions so auto-generated split codes do not block the workflow.
    if enrich_subcodes:
        sc_cb = enrich_codebook_with_ai(
            client, sc_cb, ctx,
            fill_missing_with_fallback=True,
            fallback_parent_code=parent_code
        )
        display_codebook(sc_cb)
    else:
        # This branch is reached only when all definitions were supplied manually
        # because missing definitions are blocked above if enrich_subcodes=False.
        sc_cb = _apply_definition_fallbacks(
            sc_cb, survey_context=ctx, parent_code=parent_code,
            reason='deterministic_fallback_without_ai_enrichment'
        )

    # 6. Re-code ONLY the affected rows
    #
    # When no sub-code matches a re-coded row, route the row into the top-level
    # language-specific catch-all code if one exists in the main codebook.
    # Without this, a response previously coded under the parent could be left
    # silently uncoded after the split. Matches the safety net already used by
    # apply_codes_to_dataframe.
    pc = f'code_{parent_code}'
    if pc not in df.columns: raise KeyError(f'"{pc}" not found. Run coding first.')
    aff = df[df[pc]==1].index
    if len(aff) == 0:
        logger.warning(f'No rows assigned to "{parent_code}".'); 
    else:
        logger.info(f'Re-coding {len(aff)} rows...')
        for sc in sc_cb['codes']:
            df[f"code_{sc['name']}"] = 0
            df[f"conf_{sc['name']}"] = 0.0

        other_code = get_other_code_name(codebook)
        other_col = f"code_{other_code}" if other_code else None
        other_conf_col = f"conf_{other_code}" if other_code else None
        unmatched_fallbacks = 0

        # CODING_MODEL (via _code_batch default): re-coding is structured classification
        res = _code_batch(client, df.loc[aff, text_column].tolist(), sc_cb)
        for idx, rv in zip(aff, res):
            assigned = 0
            if not isinstance(rv, dict):
                rv = {'codes': []}
            for ci in rv.get('codes', []):
                if not isinstance(ci, dict):
                    continue
                code_name = str(ci.get('code', '')).strip()
                if not code_name:
                    continue
                c_ = f"code_{code_name}"
                cf = f"conf_{code_name}"
                if c_ in df.columns:
                    conf_value = _safe_confidence(ci.get('confidence', 0.0))
                    df.at[idx, c_] = 1
                    df.at[idx, cf] = conf_value
                    ci['confidence'] = conf_value
                    assigned += 1
            if assigned == 0 and other_col and other_col in df.columns:
                df.at[idx, other_col] = 1
                df.at[idx, other_conf_col] = OTHER_FALLBACK_CONFIDENCE
                unmatched_fallbacks += 1

        if unmatched_fallbacks:
            logger.warning(
                f'{unmatched_fallbacks} re-coded row(s) matched no sub-code; '
                f'routed to "{other_code}".'
            )

    # 7. Preserve or remove parent columns
    if preserve_parent:
        logger.info(f'Parent code preserved: "{parent_code}"')
    else:
        df.drop(columns=[c for c in [f'code_{parent_code}', f'conf_{parent_code}']
                         if c in df.columns], inplace=True)
        logger.info(f'Parent code removed/replaced: "{parent_code}"')

    # 8. Update codebook
    #    preserve_parent=True  -> keep parent and insert sub-codes immediately after it
    #    preserve_parent=False -> old behavior: replace parent with sub-codes
    new_c, ins = [], False
    for code in codebook['codes']:
        if code['name'] == parent_code:
            if preserve_parent:
                new_c.append(code)
                new_c.extend(sc_cb['codes'])
            else:
                new_c.extend(sc_cb['codes'])
            ins = True
        else:
            new_c.append(code)
    if not ins:
        raise ValueError(f'"{parent_code}" not found during codebook update.')
    upd_cb = {**codebook, 'codes': new_c,
              'version': codebook.get('version', '1.0') + '+split'}
    _validate_codebook(upd_cb)
    logger.info(
        f'Codebook: "{parent_code}" preserved={preserve_parent}; '
        f'added sub-codes={final_sc}'
    )

    # 9. Rebuild summary columns + generate reports
    df = add_codes_column(df, upd_cb)
    outputs = generate_reports(df, upd_cb, output_dir,
                                text_column=text_column, output_file=output_file)
    logger.info('Split complete.')
    return df, upd_cb, outputs



def split_codes(df, codebook, client, split_configs,
                preserve_parent=False, survey_context='', text_column=None,
                output_dir='reports_s4/', output_file='split.xlsx'):
    """
    Split one or more parent codes in sequence. By default, selected parent codes are replaced by their subcodes.

    split_configs is a list of dictionaries. Each dictionary can contain:
      - parent_code              required
      - subcodes                 optional list; [] lets AI propose names
      - subcode_ids              optional list
      - subcode_definitions      optional list
      - n_subcodes_proposed      optional int, used when subcodes=[]
      - enrich                   optional bool

    Example:
    split_configs = [
        {'parent_code': 'Parent code1', 'subcodes': ['P1-A', 'P1-B']},
        {'parent_code': 'Parent code2', 'subcodes': [], 'n_subcodes_proposed': 3}
    ]
    """
    if not isinstance(split_configs, list) or not split_configs:
        raise ValueError('S4_SPLIT_CONFIGS must be a non-empty list of split rule dictionaries.')

    import copy, os

    df_out = df.copy()
    cb_out = copy.deepcopy(codebook)
    all_outputs = []

    for i, cfg in enumerate(split_configs, start=1):
        if not isinstance(cfg, dict):
            raise ValueError(f'Split config #{i} must be a dictionary.')

        parent_code = _clean_code_name(cfg.get('parent_code', ''))
        if not parent_code:
            raise ValueError(f'Split config #{i} is missing parent_code.')

        step_output_file = output_file
        if len(split_configs) > 1:
            stem, ext = os.path.splitext(output_file)
            step_output_file = f'{stem}_{i:02d}_{_slugify_for_code_id(parent_code, default="parent").lower()}{ext or ".xlsx"}'

        logger.info(f'Running split config #{i}/{len(split_configs)} for parent code "{parent_code}"')

        df_out, cb_out, outputs = split_code(
            df=df_out,
            codebook=cb_out,
            client=client,
            parent_code=parent_code,
            subcodes=cfg.get('subcodes', []),
            subcode_ids=cfg.get('subcode_ids', []),
            subcode_definitions=cfg.get('subcode_definitions', []),
            n_subcodes_proposed=cfg.get('n_subcodes_proposed', 3),
            enrich_subcodes=cfg.get('enrich', True),
            survey_context=survey_context,
            text_column=text_column,
            output_dir=output_dir,
            output_file=step_output_file,
            preserve_parent=cfg.get('preserve_parent', preserve_parent)
        )
        all_outputs.append(outputs)

    final_output_file = output_file
    if len(split_configs) > 1:
        final_output_file = output_file

    final_outputs = generate_reports(
        df_out, cb_out, output_dir,
        text_column=text_column or TEXT_COL,
        output_file=final_output_file
    )
    all_outputs.append({'final_combined_output': final_outputs})

    return df_out, cb_out, all_outputs


### 4-C · Run Section 4

In [124]:
# Save the state that Section 4 is about to modify, so the split can be undone.
set_pipeline_baseline(
    'pre_s4', df_current, codebook_current,
    metadata={
        'section': 'Before Section 4',
        'description': 'State immediately before the latest Section 4 split run.',
        'split_configs': S4_SPLIT_CONFIGS,
        'preserve_parent': S4_PRESERVE_PARENT,
    }
)

df_current, codebook_current, outputs_s4 = split_codes(
    df=df_current,
    codebook=codebook_current,
    client=client,
    split_configs=S4_SPLIT_CONFIGS,
    preserve_parent=S4_PRESERVE_PARENT,
    survey_context=SURVEY_CONTEXT,
    output_dir=S4_OUTPUT_DIR,
    output_file=S4_OUTPUT_FILE
)

# Save the post-split state too, so it can be restored after later merge experiments.
set_pipeline_baseline(
    'post_s4', df_current, codebook_current,
    metadata={
        'section': 'After Section 4',
        'description': 'State immediately after the latest Section 4 split run.',
        'split_configs': S4_SPLIT_CONFIGS,
        'preserve_parent': S4_PRESERVE_PARENT,
        'outputs': outputs_s4,
    }
)

print('\n✅ Section 4 complete!')
print(f'   Parent preserved: {S4_PRESERVE_PARENT}  # False means parent was replaced by subcodes')
print(f'   Split rules     : {[cfg["parent_code"] for cfg in S4_SPLIT_CONFIGS]}')
print(f'   Codes now       : {[c["name"] for c in codebook_current["codes"]]}')
print(f'   Files           : {outputs_s4}')
print('   Undo split      : run reset_pipeline_state("pre_s4")')
print('   Restore split   : run reset_pipeline_state("post_s4")')


01:12:12 - INFO - Codebook saved → checkpoints\PREF1_DEGUSTATION_AM_FESTI237\pre_s4__codebook.json
01:12:12 - INFO - Pipeline checkpoint saved → pre_s4 (checkpoints\PREF1_DEGUSTATION_AM_FESTI237)
01:12:12 - INFO - Running split config #1/1 for parent code "Goût agréable"
01:12:12 - INFO - ============================================================
01:12:12 - INFO - SECTION 4: Splitting "Goût agréable"
01:12:12 - INFO - ============================================================


✅ Reset baseline set: pre_s4 — 40 rows, 10 codes


01:12:13 - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
01:12:13 - INFO - LLM usage | model=gpt-5.4-mini | prompt=741 | completion=24 | total=765 | cached=0
01:12:13 - INFO - AI proposed sub-codes: ['Goût et saveur', 'Parfum et odeur', 'Aspect et équilibre']
01:12:13 - INFO - Sub-code IDs: [('C001_GOUT_ET_SAVEUR', 'Goût et saveur'), ('C001_PARFUM_ET_ODEUR', 'Parfum et odeur'), ('C001_ASPECT_ET_EQUILIBRE', 'Aspect et équilibre')]
01:12:13 - INFO - Enriching 3 codes (1 API call)...
01:12:18 - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
01:12:18 - INFO - LLM usage | model=gpt-4.1-mini | prompt=334 | completion=574 | total=908 | cached=0
01:12:18 - INFO - Enrichment complete. Matched definitions for: ['Goût et saveur', 'Parfum et odeur', 'Aspect et équilibre']
01:12:18 - INFO - Re-coding 34 rows...



CODEBOOK  version=split-draft  codes=3  multi_label=True  has_code_ids=True

[1] [C001_GOUT_ET_SAVEUR] Goût et saveur
    Definition : Ce code regroupe les mentions liées à la perception gustative de la boisson, incluant les saveurs spécifiques et l'appréciation du goût général.
    Include    : Références explicites au goût sucré, amer, acide, ou autres saveurs spécifiques.; Commentaires sur la qualité ou la richesse du goût.; Expressions d'une préférence liée à la saveur de la boisson.
    Exclude    : Mentions portant uniquement sur l'odeur ou le parfum sans lien avec le goût.; Commentaires sur le prix ou la marque sans référence au goût.
    Example    : J'aime cette boisson parce qu'elle a un goût sucré naturel qui me plaît....
    Parent     : Goût agréable
    Parent ID  : C001

[2] [C001_PARFUM_ET_ODEUR] Parfum et odeur
    Definition : Ce code concerne les appréciations liées à l'odeur ou au parfum dégagé par la boisson, qui influencent la préférence du consommateur.
    Incl

01:12:23 - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
01:12:23 - INFO - LLM usage | model=gpt-4.1-mini | prompt=470 | completion=478 | total=948 | cached=0
01:12:24 - INFO - Coded 10/34 unique
01:12:29 - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
01:12:29 - INFO - LLM usage | model=gpt-4.1-mini | prompt=440 | completion=546 | total=986 | cached=0
01:12:29 - INFO - Coded 20/34 unique
01:12:35 - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
01:12:35 - INFO - LLM usage | model=gpt-4.1-mini | prompt=451 | completion=618 | total=1069 | cached=0
01:12:36 - INFO - Coded 30/34 unique
01:12:38 - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
01:12:38 - INFO - LLM usage | model=gpt-4.1-mini | prompt=309 | completion=196 | total=505 | cached=0
01:12:38 - INFO - Coded 34/34 unique
01:12:38 - INFO - Parent code removed/replaced: "Go

✅ Reset baseline set: post_s4 — 40 rows, 12 codes

✅ Section 4 complete!
   Parent preserved: False  # False means parent was replaced by subcodes
   Split rules     : ['Goût agréable']
   Codes now       : ['Goût et saveur', 'Parfum et odeur', 'Aspect et équilibre', 'Rafraîchissement', 'Prix abordable', 'Disponibilité', 'Marque connue / fiable', 'Habitude / fidélité', 'Effet énergisant', 'Socialisation', 'Santé / Naturalité', 'Autre / Non classifiable']
   Files           : [{'frequency_chart': 'reports_s4\\PREF1_DEGUSTATION_AM_FESTI237_frequencies.png', 'cooccurrence_chart': 'reports_s4\\PREF1_DEGUSTATION_AM_FESTI237_cooccurrence.png', 'summary_report': 'reports_s4\\PREF1_DEGUSTATION_AM_FESTI237_s4_split.xlsx'}, {'final_combined_output': {'frequency_chart': 'reports_s4\\PREF1_DEGUSTATION_AM_FESTI237_frequencies.png', 'cooccurrence_chart': 'reports_s4\\PREF1_DEGUSTATION_AM_FESTI237_cooccurrence.png', 'summary_report': 'reports_s4\\PREF1_DEGUSTATION_AM_FESTI237_s4_split.xlsx'}}]
   Und

### 4-D · Inspect split results

### 4-D-1 · Frequency summary after split

In [125]:
freq_s4 = calculate_frequencies(df_current, codebook_current)
print('Frequencies after split:'); print(freq_s4.to_string(index=False))


Frequencies after split:
                 code_id                     code  count  percentage
     C001_GOUT_ET_SAVEUR           Goût et saveur     30        75.0
    C001_PARFUM_ET_ODEUR          Parfum et odeur     12        30.0
C001_ASPECT_ET_EQUILIBRE      Aspect et équilibre      8        20.0
                    C002         Rafraîchissement      4        10.0
                    C010 Autre / Non classifiable      4        10.0
                    C009       Santé / Naturalité      2         5.0
                    C003           Prix abordable      0         0.0
                    C004            Disponibilité      0         0.0
                    C006      Habitude / fidélité      0         0.0
                    C005   Marque connue / fiable      0         0.0
                    C008            Socialisation      0         0.0
                    C007         Effet énergisant      0         0.0


### 4-D-2 · Review sub-code assignments

In [126]:
print('Reviewing rows assigned to the new sub-codes. If S4_PRESERVE_PARENT=False, parent columns are intentionally removed.')
for cfg in S4_SPLIT_CONFIGS:
    parent = cfg['parent_code']
    sub_names = [c['name'] for c in codebook_current['codes']
                 if c.get('parent_code') == parent]
    sub_cols  = [f'code_{sc}' for sc in sub_names if f'code_{sc}' in df_current.columns]

    if not sub_cols:
        print(f'No sub-code columns found for parent "{parent}".')
        continue

    mask = df_current[sub_cols].sum(axis=1) > 0
    parent_col = f'code_{parent}'
    display_cols = ['response_id', 'original_text', 'codes']
    if parent_col in df_current.columns:
        display_cols.append(parent_col)
    display_cols += sub_cols

    print(f'{mask.sum()} responses re-coded into sub-codes of "{parent}"')
    display(df_current.loc[mask, display_cols].head(10))


Reviewing rows assigned to the new sub-codes. If S4_PRESERVE_PARENT=False, parent columns are intentionally removed.
34 responses re-coded into sub-codes of "Goût agréable"


,response_id,original_text,codes,code_Goût et saveur,code_Parfum et odeur,code_Aspect et équilibre
0,1,Parce que son goût est bien sucré et c'est raf...,Goût et saveur | Rafraîchissement,1,0,0
1,2,It taste good and it's very refresshing,Goût et saveur | Rafraîchissement,1,0,0
2,3,Parce que j'aime son goût de grenadine et c'es...,Goût et saveur,1,0,0
3,4,The taste is so sweet,Goût et saveur,1,0,0
4,5,"Parce que j'aime son goût de grenadine, son go...",Goût et saveur,1,0,0
5,6,C'est rafraîchissant et son goût est bon et c'...,Goût et saveur | Rafraîchissement,1,0,0
7,8,"Le goût est aromatisé, le parfum sent bon , el...",Goût et saveur | Parfum et odeur,1,1,0
8,9,Je préfère cette boisson parce que son goût es...,Goût et saveur,1,0,0
10,10,À cause de son odeur de sucre et côté arrière ...,Goût et saveur | Parfum et odeur,1,1,0
11,11,Cette boisson est équilibrée dans tout le gaz...,Aspect et équilibre,0,0,1


---
## SECTION 5 · Merge Code Groups into New Codes
*Use when one or more groups of codes are too similar or overlap too much to justify keeping separate.*  
*Prerequisite: a coded `(df_current, codebook_current)` pair from Section 2, 3, or a previous Section 4/5.*

### When to merge
- High off-diagonal values between codes in the co-occurrence heatmap
- Code names feel like synonyms or partial aspects of the same concept
- Individual code frequencies are too small to be analytically meaningful

### What this section does
```
(df_current, codebook_current)  +  one or more merge groups
  ↓
  Example:
    ['parent code1', 'parent code2'] → 'merge1'
    ['parent code5', 'parent code7', 'parent code8'] → 'merge2'
  ↓
  For each merge group:
       • merged_name = ""        → AI proposes the merged code name
       • merged_code_id = ""     → notebook auto-generates a stable ID when the codebook uses IDs
       • merged_definition = ""  → AI writes the definition when enrich=True
  ↓
  Deterministic OR-merge: merged = max(source_cols)
       No re-coding needed — any response relevant to a source code is relevant to the merged code.
  ↓
  Drop source columns; add merged code column at first source position
  ↓
  generate_reports()
  ↓
df_current, codebook_current updated in place → chain into Section 4 or another Section 5
```

### Code ID behavior
- Existing user-provided code IDs are preserved.
- If the current codebook has IDs, each merged code also gets an ID.
- Manual `merged_code_id` is used when provided.
- A missing merged ID is generated deterministically from the merged code name, then collision-checked.

### Why no API call for the merge itself
The merge is a logical union: `merged_code = source_code_1 OR source_code_2 OR ...`. The LLM is only needed if the merged code name or definition is missing.


### 5-A · Section 5 configuration

In [127]:
# df_current and codebook_current flow in from Section 4-C (or 2-C / 3-C).
# For reset/restore commands and checkpoint rules, see Section 0-G.

# Define one or more merge groups.
# - codes_to_merge: existing code names to combine
# - merged_name: new merged code name. Leave blank ('') to let AI propose a name.
# - merged_code_id: optional. Leave blank to auto-generate when code IDs are used.
# - merged_definition: optional. Leave blank with enrich=True so AI generates it.
# - enrich: True fills missing definition using AI.
S5_MERGE_CONFIGS = [
    {
        'codes_to_merge': ['Effet énergisant', 'Santé / Naturalité'],
        'merged_name': '',
        'merged_code_id': '',
        'merged_definition': '',
        'enrich': True,
    },
    # Example for multiple merge groups:
    # {
    #     'codes_to_merge': ['parent code1', 'parent code2'],
    #     'merged_name': 'merge1',
    #     'merged_code_id': '',
    #     'merged_definition': '',
    #     'enrich': True,
    # },
    # {
    #     'codes_to_merge': ['parent code5', 'parent code7', 'parent code8'],
    #     'merged_name': 'merge2',
    #     'merged_code_id': '',
    #     'merged_definition': '',
    #     'enrich': True,
    # },
]

S5_OUTPUT_DIR  = 'reports_s5/'
S5_OUTPUT_FILE = FILE_STEM + '_s5_merge.xlsx'


### 5-B · Merge function

In [128]:
def merge_codes(df, codebook, client, codes_to_merge,
                merged_name='', merged_code_id='', merged_definition='', enrich_merged=True,
                survey_context='', text_column=None,
                output_dir='reports_s5/', output_file='merge.xlsx'):
    """
    Merge two or more codes into a single consolidated code.

    Manual inputs are respected when provided. Missing merge-derived fields are
    generated automatically:
      - name: AI proposes it when merged_name is blank
      - ID: deterministic notebook-generated ID when the current codebook uses IDs
      - definition: AI enrichment when enrich_merged=True

    Existing user-provided code IDs and definitions are preserved.
    """
    if text_column is None: text_column = TEXT_COL
    ctx = survey_context or SURVEY_CONTEXT
    if len(codes_to_merge) < 2: raise ValueError('Need ≥ 2 codes to merge.')
    cb_names = [c['name'] for c in codebook['codes']]
    missing  = [c for c in codes_to_merge if c not in cb_names]
    if missing: raise ValueError(f'Not in codebook: {missing}.  Available: {cb_names}')
    logger.info('='*60)
    logger.info(f'SECTION 5: Merging {codes_to_merge}')
    logger.info('='*60)

    requires_ids = codebook_requires_manual_ids(codebook)
    merged_code_id = _clean_code_id(merged_code_id)
    merged_definition = _clean_optional_text(merged_definition)

    # 1. Determine merged code name and initial definition.
    if merged_name.strip():
        fn, fd = merged_name.strip(), merged_definition
        logger.info(f'User-defined merged name: "{fn}"')
    else:
        sums = []
        for nm in codes_to_merge:
            e = next((c for c in codebook['codes'] if c['name']==nm), {})
            missing_definition = '(no definition)' if _codebook_is_english() else '(pas de définition)'
            sums.append(f"- **{_code_ref(e)}** : {e.get('definition', missing_definition)}")
        lang_name = _codebook_language_name()
        p = (
            'You are a qualitative research expert.\n\n'
            f'CODEBOOK OUTPUT LANGUAGE: {lang_name}.\n'
            f'Generate the merged code name and definition in {lang_name}.\n\n'
            f'CONTEXT: {ctx}\n\n'
            f'The following codes will be merged:\n{chr(10).join(sums)}\n\n'
            f'Propose a short name (max 5 words) and a 1-2 sentence definition in {lang_name}.\n'
            'Return ONLY valid JSON:\n'
            '{"name": "...", "definition": "..."}'
        )
        # REASONING_MODEL: naming a merged concept requires semantic synthesis
        r = chat_with_retry(
            client, model=REASONING_MODEL,
            messages=[{'role':'system','content':'Qualitative coding expert. Return ONLY valid JSON.'},
                      {'role':'user',  'content':p}],
            temperature=0.3, max_tokens=300)
        try:
            d  = json.loads(_extract_json(r.choices[0].message.content.strip()))
            fn = d.get('name','').strip()
            fd = _clean_optional_text(d.get('definition',''))
        except Exception as e:
            logger.error(f'Name proposal parse error: {e}'); fn, fd = '', ''
        if not fn: raise ValueError('Model returned no name. Set S5_MERGED_NAME.')
        if merged_definition:
            fd = merged_definition
        logger.info(f'AI proposed merged name: "{fn}"')

    # BUGFIX 8 hardening: the merged target name must not collide with any
    # existing code that is not being consumed by this merge. This protects
    # both user-provided merged_name values and AI-generated names in multi-group
    # merge runs.
    existing_target_keys = {
        _canonical_code_key(c['name'])
        for c in codebook['codes']
        if c['name'] not in codes_to_merge
    }
    if _canonical_code_key(fn) in existing_target_keys:
        raise ValueError(
            f'Merged code name "{fn}" already exists outside the source group. '
            'Choose a unique S5_MERGED_NAME / merged_name, or reset before merging.'
        )

    # 2. Determine merged code ID. Preserve a manual ID when provided; otherwise
    # auto-generate when the current codebook uses IDs.
    source_entries = [c for c in codebook['codes'] if c['name'] in codes_to_merge]
    merged_from_ids = [_clean_code_id(c.get(CODE_ID_FIELD, '')) for c in source_entries
                       if _clean_code_id(c.get(CODE_ID_FIELD, ''))]

    if requires_ids:
        existing_ids = _existing_code_ids(codebook)
        if merged_code_id:
            if merged_code_id in existing_ids:
                raise ValueError(f'S5_MERGED_CODE_ID already exists in the codebook: {merged_code_id}')
        else:
            merged_code_id = _auto_code_id_for_new_code(
                codebook, fn, prefix='MERGE', existing_ids=existing_ids
            )
        logger.info(f'Merged code ID: {merged_code_id}')
    else:
        if merged_code_id:
            logger.warning('S5_MERGED_CODE_ID was provided but the current codebook has no IDs; ignoring it.')
        merged_code_id = ''

    # 3. Require or generate definition.
    if not fd and not enrich_merged:
        raise ValueError(
            'Merge-derived code requires a definition. Set S5_MERGED_DEFINITION '
            'or set S5_ENRICH=True so AI generates the missing definition.'
        )

    # 4. Build merged code entry
    me = make_code_entry(
        fn,
        code_id=merged_code_id,
        definition=fd,
        inclusion_criteria=[],
        exclusion_criteria=[],
        example_verbatims=[],
        merged_from=codes_to_merge,
        merged_from_ids=merged_from_ids,
        source='manual_merge' if merged_name.strip() else 'ai_merge_proposal'
    )

    # 5. Optionally enrich missing merged-code definition. If AI enrichment does
    # not return a usable definition, use a conservative deterministic fallback
    # instead of blocking automatic merge-code generation.
    if enrich_merged and not _clean_optional_text(me.get('definition')):
        mini = {'codes':[me],'multi_label':codebook.get('multi_label',True)}
        mini = enrich_codebook_with_ai(
            client, mini, ctx,
            fill_missing_with_fallback=True,
            fallback_merged_from=codes_to_merge
        )
        me = mini['codes'][0]

    if not _clean_optional_text(me.get('definition')):
        mini = {'codes':[me],'multi_label':codebook.get('multi_label',True)}
        mini = _apply_definition_fallbacks(
            mini, survey_context=ctx, merged_from=codes_to_merge,
            reason='deterministic_fallback_after_ai_enrichment'
        )
        me = mini['codes'][0]
    logger.info(f'Merged code: "{fn}" — {me.get("definition","")[:80]}')

    # 6. Deterministic OR-merge (no LLM re-coding)
    df = df.copy()
    src_c = [f'code_{c}' for c in codes_to_merge]
    src_f = [f'conf_{c}' for c in codes_to_merge]
    miss  = [c for c in src_c if c not in df.columns]
    if miss: raise KeyError(f'Columns not found: {miss}. Run coding first.')
    df[f'code_{fn}'] = df[src_c].max(axis=1).astype(int)
    ef = [c for c in src_f if c in df.columns]
    df[f'conf_{fn}'] = df[ef].max(axis=1) if ef else df[f'code_{fn}'].astype(float)
    df.drop(columns=[c for c in src_c+src_f if c in df.columns], inplace=True)
    logger.info(f'OR-merge: {int(df[f"code_{fn}"].sum())} responses carry "{fn}"')

    # 7. Update codebook — replace sources with merged entry at first source position
    new_c, ins = [], False
    for code in codebook['codes']:
        if code['name'] in codes_to_merge:
            if not ins: new_c.append(me); ins = True
        else:
            new_c.append(code)
    if not ins: raise ValueError(f'None of {codes_to_merge} found in codebook.')
    upd_cb = {**codebook,'codes':new_c,
              'version':codebook.get('version','1.0')+'+merge'}
    _validate_codebook(upd_cb)
    logger.info(f'Codebook: {codes_to_merge} → "{fn}"')

    # 8. Rebuild summary columns + generate reports
    df = add_codes_column(df, upd_cb)
    outputs = generate_reports(df, upd_cb, output_dir,
                                text_column=text_column, output_file=output_file)
    logger.info('Merge complete.')
    return df, upd_cb, outputs



def merge_code_groups(df, codebook, client, merge_configs,
                      survey_context='', text_column=None,
                      output_dir='reports_s5/', output_file='merge.xlsx'):
    """
    Apply one or more merge groups sequentially.

    Each config must have:
      - codes_to_merge: list of 2+ existing code names

    Each config may also have:
      - merged_name: target merged code name; blank lets AI propose it
      - merged_code_id: optional manual code ID
      - merged_definition: optional manual definition
      - enrich: True/False, default True

    Example:
    S5_MERGE_CONFIGS = [
        {'codes_to_merge': ['parent code1', 'parent code2'], 'merged_name': 'merge1'},
        {'codes_to_merge': ['parent code5', 'parent code7', 'parent code8'], 'merged_name': 'merge2'},
    ]
    """
    if text_column is None:
        text_column = TEXT_COL
    if not merge_configs:
        raise ValueError('S5_MERGE_CONFIGS is empty. Add at least one merge group.')

    df_out = df.copy()
    cb_out = codebook
    all_outputs = []
    used_source_codes = set()
    target_names = []
    target_name_keys = set()

    for i, cfg in enumerate(merge_configs, start=1):
        if not isinstance(cfg, dict):
            raise TypeError(f'Merge config #{i} must be a dict, got {type(cfg).__name__}.')

        codes_to_merge = cfg.get('codes_to_merge', [])
        if isinstance(codes_to_merge, str):
            raise TypeError(
                f'Merge config #{i} codes_to_merge must be a list, not a string. '
                'Use ["code A", "code B"].'
            )
        if len(codes_to_merge) < 2:
            raise ValueError(f'Merge config #{i} needs at least 2 source codes.')

        duplicate_sources = [c for c in codes_to_merge if c in used_source_codes]
        if duplicate_sources:
            raise ValueError(
                f'Merge config #{i} reuses source code(s) already merged earlier: {duplicate_sources}. '
                'A source code can only be consumed by one merge group in the same run.'
            )

        merged_name = str(cfg.get('merged_name', '') or '').strip()
        merged_name_key = _canonical_code_key(merged_name) if merged_name else ''
        if merged_name and merged_name_key in target_name_keys:
            raise ValueError(f'Merge config #{i} uses duplicate merged_name: {merged_name}')
        if merged_name:
            # Prevent confusing one run where a later group tries to use a target just created earlier.
            if merged_name in used_source_codes:
                raise ValueError(f'Merge config #{i} merged_name was already used as a source earlier: {merged_name}')
            target_names.append(merged_name)
            target_name_keys.add(merged_name_key)

        group_output_file = output_file
        if len(merge_configs) > 1:
            stem, dot, ext = output_file.rpartition('.')
            group_output_file = f'{stem or output_file}_group_{i}.{ext}' if dot else f'{output_file}_group_{i}'

        logger.info(f'Running merge group {i}/{len(merge_configs)}: {codes_to_merge} → {merged_name or "AI-generated name"}')
        df_out, cb_out, outputs = merge_codes(
            df=df_out,
            codebook=cb_out,
            client=client,
            codes_to_merge=list(codes_to_merge),
            merged_name=merged_name,
            merged_code_id=cfg.get('merged_code_id', ''),
            merged_definition=cfg.get('merged_definition', ''),
            enrich_merged=bool(cfg.get('enrich', True)),
            survey_context=survey_context,
            text_column=text_column,
            output_dir=output_dir,
            output_file=group_output_file,
        )
        all_outputs.append(outputs)
        used_source_codes.update(codes_to_merge)
        if not merged_name:
            # BUGFIX 8: compare merged_from as an order-insensitive cleaned set.
            # This avoids missing the AI-generated target name if source code names
            # were normalized, stripped, or returned in a different order.
            created = [
                c['name'] for c in cb_out['codes']
                if _same_code_name_set(c.get('merged_from', []), codes_to_merge)
            ]
            if created:
                target_names.append(created[0])
                target_name_keys.add(_canonical_code_key(created[0]))

    return df_out, cb_out, all_outputs


### 5-C · Run Section 5

In [129]:
# Save the state that Section 5 is about to modify, so the merge can be undone.
set_pipeline_baseline(
    'pre_s5', df_current, codebook_current,
    metadata={
        'section': 'Before Section 5',
        'description': 'State immediately before the latest Section 5 merge run.',
        'merge_configs': S5_MERGE_CONFIGS,
    }
)

df_current, codebook_current, outputs_s5 = merge_code_groups(
    df=df_current,
    codebook=codebook_current,
    client=client,
    merge_configs=S5_MERGE_CONFIGS,
    survey_context=SURVEY_CONTEXT,
    output_dir=S5_OUTPUT_DIR,
    output_file=S5_OUTPUT_FILE,
)

# Save the post-merge state too, so it can be restored after later experiments.
set_pipeline_baseline(
    'post_s5', df_current, codebook_current,
    metadata={
        'section': 'After Section 5',
        'description': 'State immediately after the latest Section 5 merge run.',
        'merge_configs': S5_MERGE_CONFIGS,
        'outputs': outputs_s5,
    }
)

print('\n✅ Section 5 complete!')
print(f'   Codes now: {[c["name"] for c in codebook_current["codes"]]}')
print(f'   Files    : {outputs_s5}')
print('   Undo merge    : run reset_pipeline_state("pre_s5")')
print('   Restore merge : run reset_pipeline_state("post_s5")')


01:12:46 - INFO - Codebook saved → checkpoints\PREF1_DEGUSTATION_AM_FESTI237\pre_s5__codebook.json
01:12:46 - INFO - Pipeline checkpoint saved → pre_s5 (checkpoints\PREF1_DEGUSTATION_AM_FESTI237)
01:12:46 - INFO - Running merge group 1/1: ['Effet énergisant', 'Santé / Naturalité'] → AI-generated name
01:12:46 - INFO - ============================================================
01:12:46 - INFO - SECTION 5: Merging ['Effet énergisant', 'Santé / Naturalité']
01:12:46 - INFO - ============================================================


✅ Reset baseline set: pre_s5 — 40 rows, 12 codes


01:12:47 - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
01:12:47 - INFO - LLM usage | model=gpt-5.4-mini | prompt=188 | completion=81 | total=269 | cached=0
01:12:47 - INFO - AI proposed merged name: "Bienfaits et vitalité"
01:12:47 - INFO - Merged code ID: MERGE_BIENFAITS_ET_VITALITE
01:12:47 - INFO - Merged code: "Bienfaits et vitalité" — Le répondant préfère la boisson parce qu’il lui attribue des effets bénéfiques p
01:12:47 - INFO - OR-merge: 2 responses carry "Bienfaits et vitalité"
01:12:47 - INFO - Codebook: ['Effet énergisant', 'Santé / Naturalité'] → "Bienfaits et vitalité"
01:12:48 - INFO - Frequency chart → reports_s5\PREF1_DEGUSTATION_AM_FESTI237_frequencies.png
01:12:49 - INFO - Co-occurrence heatmap → reports_s5\PREF1_DEGUSTATION_AM_FESTI237_cooccurrence.png
01:12:49 - INFO - Report → reports_s5\PREF1_DEGUSTATION_AM_FESTI237_s5_merge.xlsx
01:12:49 - INFO - Merge complete.
01:12:49 - INFO - Codebook saved → checkpoints\PREF1_DEGUS

✅ Reset baseline set: post_s5 — 40 rows, 11 codes

✅ Section 5 complete!
   Codes now: ['Goût et saveur', 'Parfum et odeur', 'Aspect et équilibre', 'Rafraîchissement', 'Prix abordable', 'Disponibilité', 'Marque connue / fiable', 'Habitude / fidélité', 'Bienfaits et vitalité', 'Socialisation', 'Autre / Non classifiable']
   Files    : [{'frequency_chart': 'reports_s5\\PREF1_DEGUSTATION_AM_FESTI237_frequencies.png', 'cooccurrence_chart': 'reports_s5\\PREF1_DEGUSTATION_AM_FESTI237_cooccurrence.png', 'summary_report': 'reports_s5\\PREF1_DEGUSTATION_AM_FESTI237_s5_merge.xlsx'}]
   Undo merge    : run reset_pipeline_state("pre_s5")
   Restore merge : run reset_pipeline_state("post_s5")


### 5-D · Inspect merge results

### 5-D-1 · Frequency summary after merge

In [130]:
freq_s5 = calculate_frequencies(df_current, codebook_current)
print('Frequencies after merge:'); print(freq_s5.to_string(index=False))


Frequencies after merge:
                    code_id                     code  count  percentage
        C001_GOUT_ET_SAVEUR           Goût et saveur     30        75.0
       C001_PARFUM_ET_ODEUR          Parfum et odeur     12        30.0
   C001_ASPECT_ET_EQUILIBRE      Aspect et équilibre      8        20.0
                       C002         Rafraîchissement      4        10.0
                       C010 Autre / Non classifiable      4        10.0
MERGE_BIENFAITS_ET_VITALITE    Bienfaits et vitalité      2         5.0
                       C003           Prix abordable      0         0.0
                       C005   Marque connue / fiable      0         0.0
                       C004            Disponibilité      0         0.0
                       C006      Habitude / fidélité      0         0.0
                       C008            Socialisation      0         0.0


### 5-D-2 · Review merged-code assignments

In [131]:
merged_names = [c['name'] for c in codebook_current['codes'] if c.get('merged_from')]
if merged_names:
    for mn in merged_names:
        mc_ = f'code_{mn}'
        if mc_ not in df_current.columns:
            continue
        mask = df_current[mc_] == 1
        source_codes = next((c.get('merged_from', []) for c in codebook_current['codes'] if c['name'] == mn), [])
        print(f'\n{mask.sum()} responses carry merged code "{mn}"')
        print(f'   merged from: {source_codes}')
        display(df_current.loc[mask, ['response_id','original_text','codes']].head(10))



2 responses carry merged code "Bienfaits et vitalité"
   merged from: ['Effet énergisant', 'Santé / Naturalité']


,response_id,original_text,codes
6,7,Le taux de sucre est moins élevé que Les autre...,Bienfaits et vitalité
41,40,Ça peut soigner des malades à cause de la prés...,Bienfaits et vitalité


---
## Reference · Chaining Sections 4 and 5

Every section updates `df_current` and `codebook_current` at the end of its run cell.  
The next section's configuration cell reads from these — no extra bookkeeping needed.

### Common chaining patterns
```
Section 1  →  Section 2  →  Section 4  →  Section 5   (AI codebook → split → merge)
Section 1  →  Section 3  →  Section 5  →  Section 4   (user codebook → merge → split)
Section 1  →  Section 3  →  Section 4  →  Section 4   (split twice)
Section 1  →  Section 2  →  Section 5  →  Section 5   (merge twice)
```

### Robust reset logic

A reset now restores the **complete pipeline state**:

1. the coded dataframe, including `code_*`, `codes`, `code_ids`, `num_codes`, and `has_multi_label`
2. the matching codebook JSON
3. checkpoint metadata explaining where the checkpoint came from

The notebook automatically creates these checkpoints:

| Checkpoint | Meaning |
|---|---|
| `s2` | Clean Section 2 output before split/merge |
| `s3` | Clean Section 3 output before split/merge |
| `pre_s4` | State immediately before the latest split |
| `post_s4` | State immediately after the latest split |
| `pre_s5` | State immediately before the latest merge |
| `post_s5` | State immediately after the latest merge |

Common resets:

```python
reset_to_section3()                  # restore clean Section 3 baseline
reset_to_section2()                  # restore clean Section 2 baseline
reset_pipeline_state('pre_s4')       # undo latest split
reset_pipeline_state('post_s4')      # restore latest split result
reset_pipeline_state('pre_s5')       # undo latest merge
reset_pipeline_state('post_s5')      # restore latest merge result
reset_pipeline_state('s3', prefer='disk')  # restore after kernel restart
list_pipeline_checkpoints()          # view available memory/disk checkpoints
```

Important: `pre_s4`, `post_s4`, `pre_s5`, and `post_s5` represent the **latest** run of those sections. If you rerun Section 4 or 5, those latest checkpoints are intentionally refreshed.
